# EV7 Taxi — ถามตอบภาษาไทย (Kaggle)

รุ่น Kaggle ของ `taxi_qa_colab.ipynb` · เนื้อหาเหมือนกัน ต่างแค่ที่เก็บไฟล์ + มี chatbot 3 แบบให้เทียบท้าย notebook

**ตั้งค่าครั้งแรก**
1. kaggle.com → **Create → New Dataset** → ชื่อ `taxi-qa-data` → ลาก `taxi_qa_data.zip` ใส่ (Kaggle แตก zip ให้เอง)
   และลากโฟลเดอร์ `lora_taxi` (ดาวน์โหลดจาก Google Drive) ใส่ dataset เดียวกัน → ตั้ง **Private** → Create
2. **Create → New Notebook** → **File → Import Notebook** → เลือกไฟล์นี้
3. แผงขวา **Input → Add Input** → ค้น `taxi-qa-data` (แท็บ Your Datasets) → ➕
4. แผงขวา **Settings**
   - **Accelerator → GPU T4 x2** (ส่วน 8 · 8.9b · 10 ต้องใช้ · ฟรี 30 ชม./สัปดาห์)
   - **Internet → On** (ต้องยืนยันเบอร์โทรในบัญชี Kaggle ก่อน · ใช้ดาวน์โหลดโมเดลและ pip)
5. รันจากบนลงล่าง (`Shift + Enter`)

> 🔒 ห้ามอัปโหลด `plate_map_LOCAL.csv` · dataset ต้องเป็น **Private** · พิกัดรถเป็นข้อมูลอ่อนไหว — ขออนุมัติก่อนใช้ cloud ภายนอก

| chatbot | cell | ค่าใช้จ่าย | ต้องมี |
|---|---|---|---|
| ① LoRA ที่เทรนไว้ | 8.1–8.3 → 8.9b → 8.8 → 8.10 | ฟรี | GPU · โฟลเดอร์ `lora_taxi` ใน dataset |
| ② Qwen chat | 10 | ฟรี | GPU |
| ③ Claude | 11 | จ่ายตามใช้ | Secret `ANTHROPIC_API_KEY` |
| เทียบ 3 แบบ | 12 | — | รันแบบไหนก่อนก็ได้ |

## 0 · ตั้งค่า Kaggle — หาไฟล์ข้อมูลจาก dataset

Kaggle วางไฟล์ dataset ไว้ที่ `/kaggle/input/<ชื่อ dataset>/` (อ่านอย่างเดียว) · ไฟล์ที่เราบันทึกเองอยู่ `/kaggle/working/` (ดาวน์โหลดได้จากแท็บ Output หลังรัน)

ผลที่ถูก: `['dim_province.csv', 'dist_pip.geojson', 'ev_chargers_osm.json', 'points.parquet']` และบอกที่อยู่ `lora_taxi`

| ถ้าเจอ | แก้ |
|---|---|
| `ไม่พบ points.parquet` | ยังไม่ได้ Add Input dataset (ข้อ 3 ด้านบน) |
| `lora_taxi: ไม่พบ` | ใช้แบบ ② ③ ได้ตามปกติ · แบบ ① ต้องอัปโหลดโฟลเดอร์ `lora_taxi` เข้า dataset |

In [ ]:
import sys, zipfile, shutil
from pathlib import Path

IN_COLAB = False
CLOUD = True                                                 # บันทึกผลลง DRIVE (= /kaggle/working)
INPUT = Path("/kaggle/input")
DRIVE = Path("/kaggle/working")                              # ที่บันทึก qa_intent.joblib / lora_taxi ใหม่
DATA = DRIVE / "taxi_qa"
DATA.mkdir(exist_ok=True)

found = sorted(INPUT.rglob("points.parquet"))
if found:                                                    # Kaggle แตก zip ให้แล้ว
    for f in found[0].parent.iterdir():
        if f.is_file():
            shutil.copy(f, DATA / f.name)
else:                                                        # ยังเป็น zip
    z = sorted(INPUT.rglob("taxi_qa_data.zip"))
    assert z, "ไม่พบ points.parquet / taxi_qa_data.zip ใน /kaggle/input — Add Input dataset ก่อน"
    zipfile.ZipFile(z[0]).extractall(DATA)

cfg = sorted(INPUT.rglob("adapter_config.json"))
LORA_IN = cfg[0].parent if cfg else None                     # โฟลเดอร์ lora_taxi ที่อัปโหลดมา
print("ข้อมูลอยู่ที่", DATA, "·", sorted(p.name for p in DATA.iterdir() if p.is_file()))
print("lora_taxi:", LORA_IN or "ไม่พบ")

## 1 · โหลดข้อมูล และเตรียมตาราง `pos`

**ข้อมูลคืออะไร** — `points.parquet` คือจุด GPS ของรถแท็กซี่ 1.17 ล้านจุด (ไฟล์ parquet = ตารางแบบบีบอัด อ่านด้วย pandas ได้เร็ว)
แต่ละแถว = รถ 1 คัน ณ เวลาหนึ่ง

| คอลัมน์ | ความหมาย |
|---|---|
| `car_id` / `plate` | รหัสรถสมมติ `EV-0001` (แทนทะเบียนจริงแล้ว) · 1 รหัส = 1 คัน (มาจาก VIN) |
| `lat` `lon` | พิกัด (ละติจูด ลองจิจูด) |
| `speed` `direction` | ความเร็ว กม./ชม. · ทิศหัวรถเป็นองศา (0 = เหนือ, 90 = ตะวันออก) |
| `meter` | `F` = มีผู้โดยสาร · `E` = ว่าง |
| `signal` | `A` = GPS ปกติ · `V` = ไม่มีสัญญาณ |
| `dist` | ลำดับอำเภอที่จุดนั้นอยู่ → cell นี้แปลงเป็นชื่อ/รหัสอำเภอและจังหวัด |

cell นี้เพิ่มคอลัมน์ `status` ด้วยกฎง่าย ๆ: ไม่มีสัญญาณ → มีผู้โดยสาร → วิ่ง (เร็วกว่า 5 กม./ชม.) → นอกนั้นคือจอด
`LATEST` = เวลาล่าสุดในข้อมูล — ระบบใช้แทนคำว่า "ตอนนี้" เพราะข้อมูลเป็นชุดเก่า (ไม่ใช่ realtime)

In [ ]:
import json, re, math, random, difflib, time
from pathlib import Path
import numpy as np
import pandas as pd

OUT = MASTER = DATA                                        # ทุกไฟล์อยู่ในโฟลเดอร์เดียว (จาก zip)

pts = pd.read_parquet(OUT / "points.parquet",
                      columns=["car_id", "plate", "meter", "signal", "lat", "lon", "speed", "direction", "gps_time", "dist"])
# car_id ในชุด Colab = รหัสสมมติต่อ VIN (1 คัน = 1 รหัส) ทำโดย make_colab_bundle.py
dist = pd.DataFrame([f["properties"] for f in json.load(open(OUT / "dist_pip.geojson", encoding="utf-8"))["features"]])
# คอลัมน์ dist ใน points = ลำดับ feature ใน dist_pip.geojson (สร้างโดย build_dashboard.assign_area)
pos = pts.assign(
    dist_code=dist.pc.str[2:].values[pts["dist"]],          # TH8301 → 8301
    prov_code=dist.ppc.str[2:].values[pts["dist"]],          # TH83 → 83
    dist_th=dist.th.values[pts["dist"]],
    prov_th=dist.pth.values[pts["dist"]],
    plate_key=pts.plate.str.replace(r"[\s-]", "", regex=True),   # ทอ-4141 / ทอ 4141 → ทอ4141
)
MOVING_KMH = 5
pos["status"] = np.select(
    [pos.signal.eq("V"), pos.meter.eq("F"), pos.speed.ge(MOVING_KMH)],
    ["ไม่มีสัญญาณ", "มีผู้โดยสาร", "ว่าง·วิ่ง"], default="จอด")
pos = pos.sort_values(["car_id", "gps_time"]).reset_index(drop=True)
LATEST = pos.gps_time.max()
print(f"{len(pos):,} จุด · {pos.car_id.nunique():,} คัน · ข้อมูลล่าสุด {LATEST}")
pos.head(3)

## 2 · รายชื่อสถานที่ (gazetteer)

**ทำไมต้องมี** — เวลาผู้ใช้พิมพ์ "ภูเก็ต" ระบบต้องรู้ว่าคือจังหวัดรหัส 83 · cell นี้รวมชื่อทุกจังหวัด (77) และอำเภอ (928) ทั้งไทย อังกฤษ และแบบมีคำนำหน้า ("จังหวัด…", "อำเภอ…", "เขต…" สำหรับกรุงเทพฯ) + ชื่อเล่นที่คนใช้บ่อย (กทม · โคราช)

อยากเพิ่มชื่อเล่น → แก้ `ALIAS` เช่น `"หาดใหญ่": ...` (ใส่รหัสจังหวัด 2 หลัก)

In [ ]:
prov = dist[["ppc", "pth", "pen"]].drop_duplicates("ppc")
gaz = []                                                     # (ชื่อที่ใช้จับ, ระดับ, รหัส, ชื่อเต็มที่แสดง)
for r in prov.itertuples():
    for n in {r.pth, "จังหวัด" + r.pth, r.pen.lower()}:
        gaz.append((n, "prov", r.ppc[2:], r.pth))
for r in dist.itertuples():
    pre = "เขต" if r.ppc == "TH10" else "อำเภอ"             # "เขต" ใช้เฉพาะกรุงเทพฯ
    for n in {r.th, pre + r.th, r.en.lower()}:
        gaz.append((n, "dist", r.pc[2:], f"{r.th} ({r.pth})"))
ALIAS = {"กทม": "10", "กรุงเทพ": "10", "บางกอก": "10", "bkk": "10", "โคราช": "30", "อยุธยา": "14"}
name_of = dict(zip(prov.ppc.str[2:], prov.pth))
gaz += [(a, "prov", c, name_of[c]) for a, c in ALIAS.items()]
gaz = pd.DataFrame(gaz, columns=["name", "level", "code", "label"]).drop_duplicates("name")

# จุดกึ่งกลางจังหวัด (จาก D:\Master) ไว้หาสถานีชาร์จใกล้จังหวัด
cent = pd.read_csv(MASTER / "dim_province.csv", dtype={"province_code": str}, encoding="utf-8-sig") \
         .set_index("province_code")[["centroid_lat", "centroid_lon"]]
print(len(gaz), "ชื่อ ·", gaz.level.value_counts().to_dict())

## 3 · ดึงค่าจากคำถาม (slot) — ทะเบียน · เวลา · สถานที่

**slot** = ค่าที่ต้องรู้ก่อนไปค้นข้อมูล — ถามว่า "EV-0042 อยู่ไหน" ต้องรู้ว่าทะเบียนคือ EV-0042

| slot | วิธีหา | ตัวอย่าง |
|---|---|---|
| ทะเบียน | **regex** (รูปแบบตัวอักษร): ตัวอักษร 1–3 ตัว + ตัวเลข 3–4 หลัก มีขีด/เว้นวรรคหรือไม่ก็ได้ | `EV-0042` `EV 0042` `ทอ-4141` |
| เวลา | คำสำคัญ: ตอนนี้/ล่าสุด · วันนี้ · เมื่อวาน · `14:30` · `9 โมง` | "เมื่อวาน" → ช่วงวันก่อนหน้าวันล่าสุดของข้อมูล |
| สถานที่ | **fuzzy match**: เทียบคำถามกับทุกชื่อใน gazetteer ให้คะแนนความเหมือน 0–1 | "ภูเกต" กับ "ภูเก็ต" ได้ 0.91 → ผ่านเกณฑ์ 0.85 |

ชื่อสั้นกว่า 4 ตัวอักษรต้องตรงเป๊ะ — ไม่งั้นคำทั่วไปอย่าง "ไปไหนมา" จะถูกจับเป็นอำเภอ "สนม"

**ลองเอง:** แก้ประโยคใน list ท้าย cell แล้วรันใหม่ ดูว่าดึงค่าได้ถูกไหม

In [ ]:
PLATE_RE = re.compile(r"(?<![ก-๙A-Za-z])((?:[ก-ฮ]{1,3}|[A-Za-z]{1,2})\s*-?\s*\d{3,4})(?!\d)")
TIME_RE = re.compile(r"(\d{1,2})\s*[:.]\s*(\d{2})\s*(?:น\.?)?|(\d{1,2})\s*โมง")

def find_plate(q):
    m = PLATE_RE.search(q)
    if not m:
        return None, q
    key = re.sub(r"[\s-]", "", m.group(1)).upper()
    return key, q.replace(m.group(1), " ")

def find_time(q):
    # คืน (เวลาเริ่ม, เวลาสิ้นสุด, คำอธิบาย) · ข้อมูลเป็น batch → อ้างอิงจาก LATEST
    day0 = LATEST.normalize()
    m = TIME_RE.search(q)
    if "เมื่อวาน" in q:
        a, b, label = day0 - pd.Timedelta(days=1), day0, "เมื่อวาน"
    elif "วันนี้" in q:
        a, b, label = day0, LATEST, "วันนี้"
    else:
        a, b, label = None, LATEST, "ล่าสุด"
    if m:
        hh = int(m.group(1) or m.group(3)); mm = int(m.group(2) or 0)
        base = a if a is not None else day0
        b = base + pd.Timedelta(hours=hh, minutes=mm)
        label = f"{label if a is not None else ''} {hh:02d}:{mm:02d} น.".strip()
        q = q.replace(m.group(0), " ")
    return (a, b, label), q

def grams(s):
    return {s[i:i + 2] for i in range(len(s) - 1)}
GAZ = [(r.name, len(r.name), grams(r.name), r) for r in gaz.itertuples() if len(r.name) >= 2]

def find_place(q, min_score=0.85):
    s = q.lower(); sg = grams(s)
    best = (0.0, None)
    for n, L, g, r in GAZ:
        if n in s:                                           # ตรงเป๊ะ: ชื่อยาวกว่าชนะ
            score = 1.0 + L / 100
        elif L < 4 or len(g & sg) < 0.4 * len(g):           # ชื่อสั้นต้องตรงเป๊ะ (fuzzy จับคำทั่วไปผิด) · กรองเร็วด้วยคู่ตัวอักษร
            continue
        else:
            score = max(difflib.SequenceMatcher(None, n, s[i:i + w]).ratio()   # ช่วงยาวใกล้เคียงชื่อ (พิมพ์ขาด/เกิน 1 ตัว)
                        for w in (L - 1, L, L + 1) for i in range(0, max(1, len(s) - w + 1)))
        if r.level == "prov":
            score += 0.001                                   # เสมอกัน → เลือกจังหวัด
        if score > best[0]:
            best = (score, r)
    if best[0] < min_score:
        return None
    r = best[1]
    return {"level": r.level, "code": r.code, "label": r.label, "matched": r.name, "score": round(min(best[0], 1.0), 2)}

def extract(q):
    plate, rest = find_plate(q)
    when, rest = find_time(rest)
    place = find_place(rest)
    return {"plate": plate, "when": when, "place": place}

for q in ["ตอนนี้รถคันไหนอยู่ภูเกตบ้าง", "EV-0042 อยู่ไหน", "เมื่อวาน EV 0042 ไปไหนมาบ้าง", "รถในเขตจตุจักรตอน 9 โมง"]:
    e = extract(q)
    print(q, "→", e["plate"], "|", e["when"][2], "|", e["place"] and (e["place"]["label"], e["place"]["score"]))

## 4 · สร้างชุดข้อมูลเทรน

**intent** = ประเภทของคำถาม (ถามหาอะไร) — ระบบนี้รู้จัก 8 แบบ:
`cars_in_area` รถในพื้นที่ · `car_status` สถานะรถคันเดียว · `car_track` เส้นทาง · `out_of_zone` รถนอกเขต · `fleet_stats` สถิติภาพรวม · `nearest_charger` จุดชาร์จ · `action_request` ขอให้สั่งการรถ (ปฏิเสธ) · `other` ไม่เกี่ยวข้อง

**ทำไมต้องสังเคราะห์** — ยังไม่มีคำถามจริงจากผู้ใช้ จึงเขียน "แม่แบบ" (template) ไว้หลายแบบต่อ intent แล้วสุ่มเติมชื่อจังหวัด ทะเบียน เวลา คำสุภาพ และจงใจพิมพ์ผิดบ้าง → ได้ ~2,300 ประโยค

**mask** — ก่อนเทรน เปลี่ยนชื่อจริงเป็นคำกลาง (`ภูเก็ต` → `สถานที่` · `EV-0042` → `ทะเบียนรถ`) ให้โมเดลเรียน**รูปประโยค** ไม่ใช่จำชื่อจังหวัด

**จะเพิ่มคำถามแบบใหม่** → เพิ่มประโยคใน dict `T` ใต้ intent ที่ตรง (ใช้ `{place}` `{plate}` `{time}` เป็นช่องเติม) แล้วรันส่วน 4–5 ใหม่

In [ ]:
T = {
 "cars_in_area": [
   "ตอนนี้รถคันไหนอยู่{place}บ้าง", "มีรถกี่คันอยู่ที่{place}", "{place}มีรถอยู่กี่คัน", "รถที่อยู่ใน{place}ตอนนี้",
   "ขอรายชื่อรถใน{place}", "{place}ตอนนี้มีแท็กซี่ไหม", "แท็กซี่ใน{place}มีกี่คัน", "รถว่างใน{place}มีกี่คัน",
   "มีรถวิ่งแถว{place}บ้างไหม", "ใครอยู่{place}บ้าง", "เช็ครถในพื้นที่{place}", "cars in {place_en}",
   "รถคันไหนจอดอยู่{place}", "{place}มีรถมีผู้โดยสารกี่คัน", "ขอดูรถทั้งหมดที่อยู่{place}",
   "จำนวนรถที่อยู่ใน{place}", "{place}มีกี่คัน", "รถเราที่อยู่{place}ตอนนี้มีใครบ้าง", "list รถใน{place}"],
 "car_status": [
   "{plate} อยู่ไหน", "รถ {plate} ตอนนี้อยู่ที่ไหน", "ตำแหน่งล่าสุดของ {plate}", "{plate} วิ่งอยู่หรือจอด",
   "เช็คสถานะรถ {plate}", "ทะเบียน {plate} อยู่จังหวัดอะไร", "{plate} มีผู้โดยสารไหม", "where is {plate}",
   "รถ {plate} ตอนนี้เป็นยังไง", "{plate} ขับไปทางไหน", "หา {plate} ให้หน่อย", "{plate} อยู่แถวไหนแล้ว",
   "{plate} ไปถึงไหนแล้ว", "{plate} ตอนนี้อยู่จังหวัดไหน", "พิกัดรถ {plate}"],
 "car_track": [
   "{plate} ไปไหนมาบ้าง", "เส้นทางของ {plate} {time}", "{plate} วิ่งไปกี่กิโล{time}", "สรุปการเดินทางรถ {plate}",
   "{plate} ผ่านจังหวัดอะไรบ้าง", "ประวัติการวิ่งของ {plate}", "{time} {plate} ไปที่ไหนบ้าง", "{plate} วิ่งทั้งหมดกี่กิโลเมตร",
   "ย้อนดูเส้นทาง {plate}", "{plate} ออกต่างจังหวัดไหม{time}", "track {plate}"],
 "out_of_zone": [
   "{time}มีรถออกนอกเขตกี่คัน", "รถคันไหนออกนอกเขต", "ใครวิ่งออกนอกพื้นที่บ้าง", "รายการรถนอกเขต",
   "มีรถวิ่งออกต่างจังหวัดไหม", "รถที่ออกนอกกรุงเทพและปริมณฑล", "นับรถที่ออกนอกเขต{time}", "รถผิดพื้นที่มีกี่คัน",
   "มีใครขับออกนอกเขตที่อนุญาตไหม", "เช็ครถออกนอกโซน", "รถคันไหนวิ่งไปต่างจังหวัด{time}", "สรุปรถออกนอกพื้นที่{time}",
   "ใครออกนอกเขตบ้าง{time}", "มีรถหลุดโซนไหม"],
 "fleet_stats": [
   "จังหวัดไหนมีรถวิ่งเยอะที่สุด", "สรุปจำนวนรถแยกตามจังหวัด", "ตอนนี้รถออนไลน์กี่คัน", "รถมีผู้โดยสารกี่เปอร์เซ็นต์",
   "มีรถทั้งหมดกี่คัน", "พื้นที่ไหนรถเยอะสุด", "ภาพรวมรถตอนนี้", "จัดอันดับจังหวัดตามจำนวนรถ", "สถิติรถ{time}",
   "รถวิ่งอยู่กี่คันจอดกี่คัน", "ทั้งประเทศมีรถออนไลน์กี่คัน", "สรุปภาพรวมทั้งหมด{time}", "ตอนนี้มีรถวิ่งกี่คัน",
   "อัตรารถมีผู้โดยสารตอนนี้", "จังหวัดที่มีรถมากที่สุด 5 อันดับ"],
 "nearest_charger": [
   "{plate} ใกล้จุดชาร์จไหน", "สถานีชาร์จใกล้{place}", "จุดชาร์จ EV ที่ใกล้รถ {plate} ที่สุด", "หาที่ชาร์จแถว{place}",
   "แถว{place}มีปั๊มชาร์จไหม", "รถ {plate} ควรไปชาร์จที่ไหน", "ที่ชาร์จไฟใกล้ๆ{place}", "charger near {place_en}"],
 "action_request": [
   "ตัดสัญญาณรถ {plate} ให้หน่อย", "ล็อครถ {plate}", "สั่งดับเครื่อง {plate}", "ปิดรถ {plate} เลย",
   "ส่งข้อความหาคนขับ {plate}", "บล็อกรถคันนี้ {plate}", "หยุดรถ {plate}", "ล็อคล้อ {plate}", "ยึดรถ {plate}",
   "ตัดไฟรถ {plate}", "ปลดล็อครถ {plate}"],
 "other": [
   "สวัสดี", "อากาศวันนี้เป็นยังไง", "ช่วยเขียนอีเมลให้หน่อย", "ราคาหุ้นวันนี้", "คุณเป็นใคร", "ขอบคุณ",
   "เล่าเรื่องตลกหน่อย", "แปลภาษาอังกฤษให้หน่อย", "วันนี้วันอะไร", "กินอะไรดี", "ราคาน้ำมันวันนี้",
   "บอกข่าวล่าสุด", "ช่วยคิดเลขให้หน่อย", "เพลงเพราะๆ"],
}
TIMES = ["", "วันนี้", "เมื่อวาน", "ตอนนี้", "ตอน 9 โมง", "เมื่อ 14:30"]
plates = pos.plate.drop_duplicates().sample(300, random_state=1).tolist()
places = gaz[gaz.level == "prov"].sample(frac=1, random_state=1)
pen = dict(zip(prov.ppc.str[2:], prov.pen))

PRE = ["", "", "", "ขอ", "ช่วยบอกหน่อย ", "อยากรู้ว่า ", "รบกวนเช็ค ", "แอดมิน "]
POST = ["", "", "", " ครับ", " คะ", " หน่อย", " ด่วน", " ?", " นะ"]

def typo(s, p=0.15):
    # ทำให้พิมพ์ผิดเล็กน้อย: ลบ/สลับ/ซ้ำ 1 ตัวอักษร
    if random.random() > p or len(s) < 4:
        return s
    i = random.randrange(1, len(s) - 1); k = random.choice("dsr")
    return s[:i] + s[i + 1:] if k == "d" else s[:i] + s[i + 1] + s[i] + s[i + 2:] if k == "s" else s[:i] + s[i] + s[i:]

random.seed(7)
rows = []
for intent, tpls in T.items():
    for ti, t in enumerate(tpls):
        for _ in range(25):
            pl = places.sample(1, random_state=random.randrange(10**6)).iloc[0]
            q = t.format(place=typo(pl["name"]), place_en=pen.get(pl.code, pl["name"]),
                         plate=random.choice(plates), time=random.choice(TIMES))
            q = random.choice(PRE) + q + random.choice(POST)    # คำสุภาพ/คำเสริม ให้ประโยคหลากหลาย
            rows.append((re.sub(r"\s+", " ", q).strip(), intent, f"{intent}#{ti}"))
train_df = pd.DataFrame(rows, columns=["q", "intent", "tpl"]).drop_duplicates("q")

def mask(q):
    # แทนค่าที่ดึงได้ด้วยคำกลาง ก่อนส่งเข้า classifier
    e = extract(q)
    _, rest = find_plate(q)
    if e["plate"]:
        rest = rest + " ทะเบียนรถ"
    _, rest = find_time(rest)
    if e["place"]:
        rest = rest.lower().replace(e["place"]["matched"], " สถานที่ ")
        if "สถานที่" not in rest:
            rest += " สถานที่"
    return re.sub(r"\s+", " ", rest).strip()

train_df["masked"] = train_df.q.map(mask)
print(len(train_df), "ประโยค ·", train_df.intent.value_counts().to_dict())
train_df.sample(6, random_state=3)[["q", "masked", "intent"]]

## 5 · เทรนตัวจัดประเภทคำถาม

**ขั้นที่ 1 — แปลงประโยคเป็นตัวเลข (embedding)**
คอมพิวเตอร์เข้าใจตัวเลข ไม่เข้าใจภาษา · **backbone** (`paraphrase-multilingual-MiniLM`) เป็นโมเดลที่ฝึกมาแล้วกับข้อความหลายภาษา แปลงประโยคเป็นเวกเตอร์ 384 ตัวเลข — ประโยคความหมายคล้ายกันจะได้ตัวเลขใกล้กัน
เป็นตัวเดียวกับที่ใช้จัดหมวดสินค้า ITEC · ครั้งแรกบน Colab จะดาวน์โหลด ~470 MB (1–2 นาที)

**ขั้นที่ 2 — เทรน classifier (Logistic Regression)** บนเวกเตอร์เหล่านั้น ให้รู้ว่าเวกเตอร์แบบไหนเป็น intent อะไร — ใช้เวลาไม่กี่วินาที ไม่ต้องใช้ GPU

**อ่านผลยังไง (ตาราง classification report)**
- **precision** — ที่โมเดลทายว่าเป็น X ถูกกี่ %
- **recall** — ข้อที่เป็น X จริง โมเดลหาเจอกี่ %
- **f1** — ค่าเฉลี่ยของสองตัวบน · **accuracy** — ถูกกี่ % ของทั้งหมด

ชุดทดสอบใช้ **template ที่โมเดลไม่เคยเห็น** (เหมือนผู้ใช้พิมพ์รูปประโยคใหม่) — คะแนนจึงต่ำกว่าการสุ่มแบ่งธรรมดาแต่ใกล้ความจริงกว่า · intent ที่ recall ต่ำ = ต้องเพิ่ม template แบบนั้น

cell ถัดไปทดสอบกับ **คำถามที่เขียนเอง 20 ข้อ** (`HAND`) แล้วแสดงข้อที่ทายผิด

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

BACKBONE = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"   # ตัวเดียวกับ scripts/itec/category (cache ไว้แล้ว)
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
enc = SentenceTransformer(BACKBONE, device=DEVICE)          # Colab ดาวน์โหลดโมเดลเองครั้งแรก (~470 MB)
print("device:", DEVICE)

t0 = time.time()
X = enc.encode(train_df.masked.tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
print(f"encode {len(X):,} ประโยค {time.time() - t0:.0f} วินาที")

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)       # แบ่งตาม template
tr, te = next(gss.split(X, train_df.intent, groups=train_df.tpl))
clf = LogisticRegression(C=8, max_iter=2000, class_weight="balanced").fit(X[tr], train_df.intent.iloc[tr])
print(classification_report(train_df.intent.iloc[te], clf.predict(X[te]), digits=3, zero_division=0))

In [ ]:
# เทรนใหม่ด้วยข้อมูลทั้งหมดสำหรับใช้งาน + ชุดทดสอบที่เขียนเอง (ไม่ได้มาจาก template)
clf = LogisticRegression(C=8, max_iter=2000, class_weight="balanced").fit(X, train_df.intent)

HAND = [
 ("ภูเก็ตตอนนี้มีรถเรามั้ย", "cars_in_area"), ("แถวบางนามีแท็กซี่ว่างกี่คันครับ", "cars_in_area"),
 ("ขอลิสต์รถที่อยู่เชียงใหม่", "cars_in_area"), ("นนทบุรีมีรถกี่คันตอนนี้", "cars_in_area"),
 ("ทอ-4141 ตอนนี้ไปถึงไหนแล้ว", "car_status"), ("รถทะเบียน ทอ 8675 จอดอยู่หรือเปล่า", "car_status"),
 ("ช่วยหาตำแหน่ง E-1234", "car_status"),
 ("เมื่อวานทอ-4141 วิ่งไปไหนบ้าง", "car_track"), ("ทอ-8571 วันนี้วิ่งไปกี่โล", "car_track"),
 ("วันนี้มีใครออกนอกเขตบ้าง", "out_of_zone"), ("รถที่วิ่งออกต่างจังหวัดวันนี้", "out_of_zone"),
 ("ตอนนี้รถวิ่งอยู่ทั้งหมดกี่คัน", "fleet_stats"), ("จังหวัดไหนรถเยอะที่สุดตอนนี้", "fleet_stats"),
 ("ทอ-4141 แบตใกล้หมด ชาร์จที่ไหนใกล้สุด", "nearest_charger"), ("ที่ชาร์จรถไฟฟ้าแถวหาดใหญ่", "nearest_charger"),
 ("ช่วยตัดไฟรถ ทอ-4141 ที", "action_request"), ("ล็อคล้อรถคันนี้ ทอ-8675", "action_request"),
 ("หวัดดีครับ", "other"), ("ฝนจะตกไหมวันนี้", "other"), ("ขอสูตรต้มยำ", "other"),
]
hand = pd.DataFrame(HAND, columns=["q", "intent"])
Xh = enc.encode(hand.q.map(mask).tolist(), normalize_embeddings=True)
proba = clf.predict_proba(Xh)
hand["pred"] = clf.classes_[proba.argmax(1)]
hand["conf"] = proba.max(1).round(2)
print(f"ถูก {(hand.pred == hand.intent).mean():.0%} ของ {len(hand)} ข้อ")
hand[hand.pred != hand.intent]

## 6 · ตัวทำงาน (executor) — ค้นข้อมูลจริง

เมื่อรู้ intent และ slot แล้ว ส่วนนี้คือ "คนไปค้นตาราง" — เป็นโค้ด pandas ธรรมดา **ตัวเลขทุกตัวในคำตอบมาจากที่นี่** (ไม่มีโมเดลไหนเดาตัวเลข)

| ฟังก์ชัน | ทำอะไร |
|---|---|
| `latest_at(t)` | ตำแหน่งล่าสุดของแต่ละคัน ณ เวลา t (นับเฉพาะจุดใน 15 นาทีก่อนหน้า — เก่ากว่านั้นถือว่าออฟไลน์) |
| `x_cars_in_area` | กรองจาก `latest_at` เฉพาะจังหวัด/อำเภอที่ถาม |
| `x_car_status` | จุดล่าสุดของรถคันนั้น |
| `x_car_track` | รวมระยะทาง (สูตร haversine = ระยะบนผิวโลก) ไม่นับช่วงที่สัญญาณหายเกิน 15 นาที |
| `x_out_of_zone` | รถที่มีจุดนอกกรุงเทพฯ และปริมณฑล |
| `x_fleet_stats` | นับรถออนไลน์ แยกสถานะ และ 5 จังหวัดที่รถเยอะสุด |
| `x_nearest_charger` | สถานีชาร์จที่ใกล้รถ (หรือใกล้กลางจังหวัด) ที่สุด 3 แห่ง |

In [ ]:
STALE = pd.Timedelta(minutes=15)
ZONE = {"10", "11", "12", "13", "73", "74"}             # กรุงเทพฯ และปริมณฑล
chargers = json.load(open(OUT / "ev_chargers_osm.json", encoding="utf-8"))
chg = pd.DataFrame([{"lat": (e.get("lat") or e["center"]["lat"]), "lon": (e.get("lon") or e["center"]["lon"]),
                     "name": e.get("tags", {}).get("name", "") or "(ไม่มีชื่อ)",
                     "op": e.get("tags", {}).get("operator") or e.get("tags", {}).get("brand") or ""}
                    for e in chargers if "lat" in e or "center" in e])

def hav_km(lat1, lon1, lat2, lon2):
    p = np.pi / 180
    a = np.sin((lat2 - lat1) * p / 2) ** 2 + np.cos(lat1 * p) * np.cos(lat2 * p) * np.sin((lon2 - lon1) * p / 2) ** 2
    return 12742 * np.arcsin(np.sqrt(a))

def latest_at(t):
    w = pos[(pos.gps_time <= t) & (pos.gps_time > t - STALE)]
    return w.groupby("car_id").tail(1)

def car_rows(plate_key):
    return pos[pos.plate_key.str.upper() == plate_key]

def x_cars_in_area(place, when):
    if not place:
        return {"error": "ไม่พบชื่อสถานที่ในคำถาม"}
    now = latest_at(when[1])
    col = "prov_code" if place["level"] == "prov" else "dist_code"
    hit = now[now[col] == place["code"]]
    return {"as_of": when[1], "place": place["label"], "n": len(hit), "by_status": hit.status.value_counts().to_dict(),
            "cars": hit[["car_id", "plate", "status", "dist_th"]].head(20).to_dict("records")}

def x_car_status(plate, when):
    r = car_rows(plate) if plate else pd.DataFrame()
    if r.empty:
        return {"error": f"ไม่พบรถทะเบียน {plate or '(ไม่ได้ระบุ)'}"}
    r = r[r.gps_time <= when[1]]
    if r.empty:
        return {"error": "ไม่มีข้อมูลก่อนเวลาที่ถาม"}
    x = r.iloc[-1]
    return {"as_of": when[1], "plate": x.plate, "car_id": x.car_id, "time": x.gps_time, "age_min": round((when[1] - x.gps_time).total_seconds() / 60),
            "status": x.status, "speed": x.speed, "heading": x.direction, "where": f"{x.dist_th}, {x.prov_th}", "lat": x.lat, "lon": x.lon}

def x_car_track(plate, when):
    r = car_rows(plate) if plate else pd.DataFrame()
    if r.empty:
        return {"error": f"ไม่พบรถทะเบียน {plate or '(ไม่ได้ระบุ)'}"}
    a = when[0] if when[0] is not None else LATEST.normalize()
    r = r[(r.gps_time >= a) & (r.gps_time <= when[1]) & (r.signal != "V")]
    if len(r) < 2:
        return {"error": "ข้อมูลในช่วงนี้ไม่พอ"}
    gap = r.gps_time.diff() <= STALE                       # ไม่นับระยะข้ามช่วงที่สัญญาณหาย
    km = hav_km(r.lat.shift(), r.lon.shift(), r.lat, r.lon)[gap].sum()
    provs = r.groupby("prov_th", sort=False).size()
    return {"plate": r.plate.iloc[0], "from": r.gps_time.iloc[0], "to": r.gps_time.iloc[-1], "km": round(float(km), 1),
            "max_speed": r.speed.max(), "provinces": list(provs.index), "start": f"{r.dist_th.iloc[0]}, {r.prov_th.iloc[0]}",
            "end": f"{r.dist_th.iloc[-1]}, {r.prov_th.iloc[-1]}"}

def x_out_of_zone(when):
    a = when[0] if when[0] is not None else LATEST.normalize()
    r = pos[(pos.gps_time >= a) & (pos.gps_time <= when[1]) & (pos.signal != "V") & ~pos.prov_code.isin(ZONE)]
    g = r.groupby(["car_id", "plate"]).agg(provinces=("prov_th", lambda s: ", ".join(s.unique()[:3])), points=("prov_th", "size"))
    g = g.sort_values("points", ascending=False).reset_index()
    top = r.groupby("prov_th").car_id.nunique().sort_values(ascending=False).head(5)
    return {"from": a, "to": when[1], "n": len(g), "cars": g.head(20).to_dict("records"), "top_prov": top.to_dict()}

def x_fleet_stats(when):
    now = latest_at(when[1])
    top = now.groupby("prov_th").size().sort_values(ascending=False).head(5)
    return {"as_of": when[1], "online": len(now), "by_status": now.status.value_counts().to_dict(), "top_prov": top.to_dict()}

def x_nearest_charger(plate, place, when, k=3):
    if plate:
        s = x_car_status(plate, when)
        if "error" in s:
            return s
        lat, lon, ref = s["lat"], s["lon"], f"รถ {s['plate']} ({s['where']})"
    elif place:
        code = place["code"][:2]
        lat, lon = cent.loc[code, "centroid_lat"], cent.loc[code, "centroid_lon"]
        ref = f"กลางจังหวัด {name_of[code]}"
    else:
        return {"error": "ต้องระบุทะเบียนรถหรือสถานที่"}
    d = chg.assign(km=hav_km(lat, lon, chg.lat, chg.lon)).nsmallest(k, "km")
    return {"ref": ref, "items": d[["name", "op", "km"]].round(1).to_dict("records"),
            "note": "ข้อมูลสถานีชาร์จจาก OpenStreetMap มีแค่บางส่วน (306 จุด)"}

## 7 · รวมเป็น `ask()` — ถามได้เลย

ลำดับใน `ask(คำถาม)`: ดึง slot (ส่วน 3) → ทาย intent (ส่วน 5) → ถ้ามั่นใจต่ำกว่า `MIN_CONF` (0.45) **ถามกลับ ไม่เดา** → เรียก executor (ส่วน 6) → สร้างประโยคตอบจากแม่แบบ

**ลองถามเอง:** เพิ่ม cell ใหม่ (ปุ่ม `+ Code`) แล้วพิมพ์
```python
print(ask("ตอนนี้รถคันไหนอยู่ชลบุรีบ้าง", verbose=True))
```
`verbose=True` จะแสดงว่าระบบเข้าใจคำถามยังไง (intent · ความมั่นใจ · slot) — ใช้ดูว่าผิดตรงไหนเวลาตอบไม่ถูก

In [ ]:
MIN_CONF = 0.45            # มั่นใจต่ำกว่านี้ = ถามกลับ ไม่เดา

def fmt_t(t):
    return pd.Timestamp(t).strftime("%d/%m %H:%M")

def answer(intent, r):
    if "error" in r:
        return f"ขออภัย {r['error']}"
    if intent == "cars_in_area":
        if not r["n"]:
            return f"ไม่มีรถออนไลน์ใน{r['place']} (ข้อมูล ณ {fmt_t(r['as_of'])})"
        st = " · ".join(f"{k} {v}" for k, v in r["by_status"].items())
        lst = ", ".join(c["plate"] for c in r["cars"][:10])
        more = f" และอีก {r['n'] - 10} คัน" if r["n"] > 10 else ""
        return f"{r['place']} มีรถออนไลน์ {r['n']} คัน ({st}) — {lst}{more} · ข้อมูล ณ {fmt_t(r['as_of'])}"
    if intent == "car_status":
        return (f"{r['plate']} อยู่ที่ {r['where']} · {r['status']} {r['speed']:.0f} กม./ชม. ทิศ {r['heading']:.0f}° "
                f"· จุดล่าสุด {fmt_t(r['time'])} ({r['age_min']} นาทีก่อนเวลาที่ถาม)")
    if intent == "car_track":
        return (f"{r['plate']} ช่วง {fmt_t(r['from'])}–{fmt_t(r['to'])} วิ่งประมาณ {r['km']} กม. (เร็วสุด {r['max_speed']:.0f} กม./ชม.) "
                f"จาก {r['start']} ถึง {r['end']} · ผ่าน {', '.join(r['provinces'])}")
    if intent == "out_of_zone":
        lst = ", ".join(f"{c['plate']} ({c['provinces']})" for c in r["cars"][:8])
        return f"ช่วง {fmt_t(r['from'])}–{fmt_t(r['to'])} มีรถออกนอกกรุงเทพฯ และปริมณฑล {r['n']} คัน" + (" · ไปมากสุด (คัน): " + ", ".join(f"{k} {v}" for k, v in r["top_prov"].items()) if r.get("top_prov") else "")
    if intent == "fleet_stats":
        st = " · ".join(f"{k} {v}" for k, v in r["by_status"].items())
        top = ", ".join(f"{k} {v}" for k, v in r["top_prov"].items())
        return f"ณ {fmt_t(r['as_of'])} รถออนไลน์ {r['online']:,} คัน ({st}) · จังหวัดที่รถเยอะสุด: {top}"
    if intent == "nearest_charger":
        lst = " · ".join(f"{i['name']}{' (' + i['op'] + ')' if i['op'] else ''} {i['km']} กม." for i in r["items"])
        return f"สถานีชาร์จใกล้{r['ref']}: {lst} — {r['note']}"
    return ""

def ask(q, verbose=False):
    e = extract(q)
    p = clf.predict_proba(enc.encode([mask(q)], normalize_embeddings=True))[0]
    intent, conf = clf.classes_[p.argmax()], float(p.max())
    if verbose:
        print(f"  intent={intent} ({conf:.2f}) plate={e['plate']} place={e['place'] and e['place']['label']} when={e['when'][2]}")
    if conf < MIN_CONF:
        return "ไม่แน่ใจว่าถามเรื่องอะไร ลองถามเช่น 'ตอนนี้รถคันไหนอยู่ภูเก็ต' หรือ 'ทอ-4141 อยู่ไหน'"
    if intent == "action_request":
        return "ระบบนี้ตอบคำถามได้อย่างเดียว ไม่สามารถสั่งการรถ (ตัดสัญญาณ/ล็อก/ดับเครื่อง) ได้"
    if intent == "other":
        return "ตอบได้เฉพาะคำถามเรื่องรถแท็กซี่ EV7 เช่น ตำแหน่งรถ รถในพื้นที่ เส้นทาง รถนอกเขต สถานีชาร์จ"
    run = {"cars_in_area": lambda: x_cars_in_area(e["place"], e["when"]),
           "car_status": lambda: x_car_status(e["plate"], e["when"]),
           "car_track": lambda: x_car_track(e["plate"], e["when"]),
           "out_of_zone": lambda: x_out_of_zone(e["when"]),
           "fleet_stats": lambda: x_fleet_stats(e["when"]),
           "nearest_charger": lambda: x_nearest_charger(e["plate"], e["place"], e["when"])}[intent]
    return answer(intent, run())

demo_plate = pos.plate.value_counts().index[0]      # ทะเบียนที่มีข้อมูลมากสุด ใช้ทดลอง
for q in ["ตอนนี้รถคันไหนอยู่ภูเกตบ้าง", f"{demo_plate} อยู่ไหน", f"วันนี้ {demo_plate} ไปไหนมาบ้าง",
          "วันนี้มีรถออกนอกเขตกี่คัน", "จังหวัดไหนมีรถเยอะที่สุด", "ที่ชาร์จแถวเชียงใหม่",
          f"ตัดสัญญาณรถ {demo_plate} ให้หน่อย", "ขอสูตรต้มยำ"]:
    print("Q:", q); print("A:", ask(q, verbose=True)); print()

In [ ]:
# บันทึกโมเดล (backbone ไม่ต้องบันทึก — โหลดจากชื่อ)
import joblib
SAVE = (DRIVE if CLOUD else DATA) / "qa_intent.joblib"     # Kaggle: /kaggle/working (ดาวน์โหลดจากแท็บ Output)
joblib.dump({"clf": clf, "backbone": BACKBONE, "min_conf": MIN_CONF, "trained": str(pd.Timestamp.now())}, SAVE)
print("บันทึก", SAVE)

### 7.1 · จัดอันดับพื้นที่ — "เขตไหนมีรถมากที่สุด"

In [ ]:
# ── 7.1 · จัดอันดับพื้นที่ — "เขตไหน / อำเภอไหน / จังหวัดไหน มีรถมากที่สุด" ──
# รันหลังส่วน 7 (และหลัง cell chatbot ถ้ามี) · ห่อ ask() และ facts_for() เดิม ไม่ต้องเทรนใหม่
import re

RANK_RE = re.compile(r"(เขต|อำเภอ|จังหวัด|พื้นที่|แขวง|โซน|ตำบล)\s*ไหน|ที่ไหน.*(มาก|เยอะ)|(มาก|เยอะ|สูง)\s*(ที่)?สุด|อันดับ|top\s*\d*", re.I)

def is_rank(q):
    return bool(RANK_RE.search(q)) and not find_plate(q)[0]     # มีทะเบียน = ถามรถคันเดียว ไม่ใช่จัดอันดับ

def x_rank_area(place, when, top=5):
    now = latest_at(when[1])
    if place and place["level"] == "dist":                     # อำเภอเดียว → ไม่มีอะไรให้จัดอันดับ ตอบจำนวนรถ
        return x_cars_in_area(place, when)
    if place:                                                  # จังหวัด → จัดอันดับเขต/อำเภอในจังหวัดนั้น
        sub, col, scope = now[now.prov_code == place["code"]], "dist_th", place["label"]
        unit = "เขต" if place["code"] == "10" else "อำเภอ"
    else:                                                      # ไม่ระบุ → จัดอันดับจังหวัดทั้งประเทศ
        sub, col, scope, unit = now, "prov_th", "ทั้งประเทศ", "จังหวัด"
    g = (sub.assign(hired=sub.status.eq("มีผู้โดยสาร"))
            .groupby(col).agg(n=("car_id", "size"), hired=("hired", "sum"))
            .sort_values("n", ascending=False).head(top))
    return {"as_of": when[1], "scope": scope, "unit": unit, "total": len(sub),
            "top": [(k, int(r.n), int(r.hired)) for k, r in g.iterrows()]}

def rank_text(r):
    if "error" in r or "top" not in r:
        return answer("cars_in_area", r)
    if not r["top"]:
        return f"ไม่มีรถออนไลน์ใน{r['scope']} (ข้อมูล ณ {fmt_t(r['as_of'])})"
    lst = " · ".join(f"{i}. {name} {n} คัน (มีผู้โดยสาร {h})" for i, (name, n, h) in enumerate(r["top"], 1))
    return f"{r['unit']}ที่มีรถมากที่สุดใน{r['scope']} (รถออนไลน์ทั้งหมด {r['total']:,} คัน): {lst} · ข้อมูล ณ {fmt_t(r['as_of'])}"

def rank_facts(q):
    e = extract(q)
    if "memory" in globals() and not e["place"] and memory.get("place") and re.search(r"ที่นั่น|จังหวัดนี้|แถวนั้น|ที่เดิม", q):
        e["place"] = memory["place"]
    r = x_rank_area(e["place"], e["when"])
    if "memory" in globals() and e["place"]:
        memory["place"] = e["place"]
    return rank_text(r)

# ห่อ ask() เดิม (รันซ้ำได้ ไม่ห่อซ้อน)
if "_ask_orig" not in globals():
    _ask_orig = ask
def ask(q, verbose=False):
    if is_rank(q):
        if verbose:
            print("  intent=area_ranking (กฎคำว่า 'ไหน…มากที่สุด')")
        return rank_facts(q)
    return _ask_orig(q, verbose)

# ห่อ facts_for() ของ chatbot ถ้ามี
if "facts_for" in globals():
    if "_facts_orig" not in globals():
        _facts_orig = facts_for
    def facts_for(q):                                          # cell 10 คืน (intent, ข้อความ)
        return ("area_ranking", rank_facts(q)) if is_rank(q) else _facts_orig(q)

for q in ["รถอยู่เขตไหนในกรุงเทพมากที่สุด", "อำเภอไหนในชลบุรีมีรถเยอะสุด", "จังหวัดไหนมีรถเยอะที่สุด",
          "ตอนนี้รถคันไหนอยู่ภูเกตบ้าง", "ที่ชาร์จใกล้ที่สุดแถวหาดใหญ่"]:
    print("Q:", q); print("A:", ask(q, verbose=True)); print()

### 7.2 · คำถามเชิงวิเคราะห์

ช่วงเวลาที่ผู้โดยสารเยอะ · ช่วงวันที่ของข้อมูล · จำนวนสถานีชาร์จในพื้นที่ · รถจอดใกล้สถานีชาร์จ · แนะนำทำเลเปิดสถานีชาร์จ ·
เดาปลายทางรถที่สัญญาณหาย · รถที่รับผู้โดยสารบ่อยสุด · สัดส่วนรถวิ่ง/จอด

**เลือกประเภทคำถามยังไง (semantic router):** แปลงคำถามเป็นเวกเตอร์ด้วย MiniLM ตัวเดิม แล้วเทียบกับตัวอย่างใน `EXAMPLES`
ถ้าใกล้ตัวอย่างประเภทใหม่มากกว่าคำถามแบบเดิม (ส่วน 4) → ใช้ตัวคำนวณใหม่ · ไม่ต้องเทรน · **ระบบเข้าใจผิด = เพิ่มตัวอย่างคำถามนั้นใน `EXAMPLES` แล้วรันใหม่**

ทุกตัวเลขคำนวณจากข้อมูลจริง · ส่วนที่เป็นการประมาณ (รถใกล้สถานีชาร์จ · ทำเล · ปลายทาง) จะบอกในคำตอบ

In [ ]:
# ── 7.2 · คำถามเชิงวิเคราะห์ — ช่วงเวลา · ช่วงข้อมูล · สถานีชาร์จ · แนะนำทำเล · เดาปลายทาง · รับผู้โดยสาร · สัดส่วน ──
# รันหลัง 7.1 · ทุกตัวเลขคำนวณจากข้อมูลจริง (pos / chg) — LLM แค่เรียบเรียง
#
# เลือกประเภทคำถามด้วย "ความหมายใกล้ตัวอย่าง" (semantic router): แปลงคำถามเป็นเวกเตอร์ด้วย MiniLM ตัวเดิม (enc)
# แล้วเทียบกับตัวอย่างของแต่ละประเภท ถ้าใกล้พอ (≥ ROUTE_MIN) ใช้ประเภทนั้น · ไม่ต้องเทรนใหม่ · อยากให้เข้าใจเพิ่ม = เพิ่มตัวอย่างใน EXAMPLES
import json
import numpy as np
import pandas as pd
from shapely.geometry import shape, Point
from shapely import STRtree

ROUTE_MIN = 0.72          # ต่ำกว่านี้ = ไม่ใช่คำถามวิเคราะห์ → ส่งต่อให้ระบบเดิม
NEAR_KM = 0.3             # "จอดใกล้สถานีชาร์จ" = ภายใน 300 ม.
TH_MONTH = ["", "ม.ค.", "ก.พ.", "มี.ค.", "เม.ย.", "พ.ค.", "มิ.ย.", "ก.ค.", "ส.ค.", "ก.ย.", "ต.ค.", "พ.ย.", "ธ.ค."]


def th_hour(h):
    # 15 → "บ่าย 3" · คนไทยพูดเวลาแบบนี้
    if h == 0: return "เที่ยงคืน"
    if h < 6: return f"ตี {h}"
    if h < 12: return f"{h} โมงเช้า"
    if h == 12: return "เที่ยง"
    if h < 16: return f"บ่าย {h - 12}"
    if h < 19: return f"{h - 12} โมงเย็น"
    return f"{h - 18} ทุ่ม"


def th_date(t):
    t = pd.Timestamp(t)
    return f"{t.day} {TH_MONTH[t.month]} {t.year}"


# ── เตรียมข้อมูลครั้งเดียว ─────────────────────────────────────────────────
_ok = pos[pos.signal != "V"]
_feat = json.load(open(OUT / "dist_pip.geojson", encoding="utf-8"))["features"]
_polys = [shape(f["geometry"]) for f in _feat]
_tree = STRtree(_polys)


def _dist_idx(lat, lon):
    hit = _tree.query(Point(lon, lat), predicate="within")
    return int(hit[0]) if len(hit) else -1


chg["dist"] = [_dist_idx(a, b) for a, b in zip(chg.lat, chg.lon)]
chg["dist_code"] = [dist.pc.iloc[i][2:] if i >= 0 else None for i in chg["dist"]]
chg["prov_code"] = [dist.ppc.iloc[i][2:] if i >= 0 else None for i in chg["dist"]]
chg["dist_th"] = [dist.th.iloc[i] if i >= 0 else None for i in chg["dist"]]
_dist_cent = pd.DataFrame({"lat": [p.centroid.y for p in _polys], "lon": [p.centroid.x for p in _polys]})

# จำนวนรถเฉลี่ยต่อชั่วโมงในแต่ละเขต/อำเภอ (นับรถไม่ซ้ำต่อชั่วโมง แล้วเฉลี่ยทุกชั่วโมงที่มีข้อมูล)
_hr = _ok.assign(hour=_ok.gps_time.dt.floor("h"))
_n_hours = _hr.hour.nunique()
_dist_load = (_hr.groupby(["dist_code", "hour"]).car_id.nunique().groupby("dist_code").sum() / _n_hours).rename("cars_avg")


# ── ตัวคำนวณ ──────────────────────────────────────────────────────────────
def x_peak_hour(when=None):
    f = _hr[_hr.meter == "F"]
    per = f.groupby("hour").car_id.nunique()                      # รถมีผู้โดยสาร (ไม่ซ้ำ) ต่อชั่วโมง
    by_h = per.groupby(per.index.hour).mean().round().astype(int)
    h = int(by_h.idxmax())
    low = int(by_h.idxmin())
    return {"text": f"ช่วงที่มีผู้โดยสารมากที่สุดคือ {th_hour(h)} ({h:02d}:00–{h:02d}:59) เฉลี่ย {by_h[h]} คันที่มีผู้โดยสาร · "
                    f"น้อยสุดคือ {th_hour(low)} ({by_h[low]} คัน) · คิดจากข้อมูล {th_date(pos.gps_time.min())} – {th_date(LATEST)} "
                    f"(ข้อมูลแค่ประมาณ 1 วันครึ่ง ยังสรุปเป็นแนวโน้มประจำไม่ได้)"}


def x_data_range(when=None):
    a, b = pos.gps_time.min(), pos.gps_time.max()
    return {"text": f"มีข้อมูลตั้งแต่ {th_date(a)} {a:%H:%M} ถึง {th_date(b)} {b:%H:%M} "
                    f"รวม {pos.car_id.nunique():,} คัน · ระบบถือว่า \"ตอนนี้\" = {b:%H:%M} ของวันที่ {th_date(b)}"}


def x_chargers_in_area(place, when=None):
    if place:
        col = "prov_code" if place["level"] == "prov" else "dist_code"
        sub = chg[chg[col] == place["code"]]
        scope = place["label"]
    else:
        sub, scope = chg, "ทั้งประเทศ"
    by = sub.dist_th.value_counts().head(5)
    top = " · ".join(f"{k} {v} แห่ง" for k, v in by.items())
    return {"text": f"{scope} มีสถานีชาร์จ EV {len(sub)} แห่ง" + (f" (มากสุด: {top})" if len(sub) else "")
                    + " — ข้อมูลจาก OpenStreetMap มีแค่บางส่วน ของจริงน่าจะมากกว่านี้"}


def x_cars_near_chargers(place, when):
    now = latest_at(when[1])
    now = now[now.status == "จอด"]
    c = chg
    if place:
        col = "prov_code" if place["level"] == "prov" else "dist_code"
        c, now = chg[chg[col] == place["code"]], now[now[col] == place["code"]]
    if c.empty:
        return {"text": "ไม่พบสถานีชาร์จในพื้นที่นี้ในข้อมูล OpenStreetMap"}
    d = hav_km(now.lat.values[:, None], now.lon.values[:, None], c.lat.values[None, :], c.lon.values[None, :])
    near = d.min(axis=1) <= NEAR_KM
    nearest = d.argmin(axis=1)[near]
    top = pd.Series(nearest).map(lambda i: f"{c.name.iloc[i]} ({c.dist_th.iloc[i]})").value_counts().head(3)
    lst = " · ".join(f"{k} {v} คัน" for k, v in top.items())
    return {"text": f"ณ {fmt_t(when[1])} มีรถจอดอยู่ใกล้สถานีชาร์จ (ภายใน {int(NEAR_KM * 1000)} ม.) ประมาณ {int(near.sum())} คัน"
                    + (f" — มากสุดที่ {lst}" if lst else "")
                    + " · เป็นการประมาณจากระยะห่าง ไม่รู้ว่ากำลังชาร์จจริงหรือไม่"}


def x_site_recommend(place, when=None):
    code = place["code"][:2] if place else "10"                    # ไม่ระบุ = กรุงเทพฯ
    d = dist[dist.ppc.str[2:] == code].assign(dist_code=lambda x: x.pc.str[2:])
    d = d.merge(_dist_load, left_on="dist_code", right_index=True, how="left").fillna({"cars_avg": 0})
    d["chargers"] = d.dist_code.map(chg.dist_code.value_counts()).fillna(0).astype(int)
    d["score"] = d.cars_avg / (d.chargers + 1)                      # รถเยอะ + สถานีน้อย = น่าเปิด
    top = d.nlargest(3, "score")
    unit = "เขต" if code == "10" else "อำเภอ"
    lst = " · ".join(f"{i}. {unit}{r.th} (รถเฉลี่ย {r.cars_avg:.0f} คัน/ชม. · สถานีชาร์จ {r.chargers} แห่ง)"
                     for i, r in enumerate(top.itertuples(), 1))
    return {"text": f"{unit}ที่น่าเปิดสถานีชาร์จใน{name_of[code]} (รถเยอะแต่สถานีน้อย): {lst} "
                    f"— เป็นคำแนะนำเบื้องต้นจากจำนวนรถและสถานีใน OpenStreetMap เท่านั้น ยังไม่ได้ดูค่าเช่า/ไฟฟ้า/ที่จอด"}


def x_lost_signal(plate, place, when):
    # รถที่สัญญาณหาย: จุดสุดท้ายเก่ากว่า 15 นาที หรือ signal = V · เดาปลายทางจากทิศ + ความเร็วล่าสุด
    last = _ok[_ok.gps_time <= when[1]].groupby("car_id").tail(1)
    lost = last[(when[1] - last.gps_time > STALE) & (last.speed >= 20)]
    if plate:
        lost = lost[lost.plate_key.str.upper() == plate]
    elif place:
        col = "prov_code" if place["level"] == "prov" else "dist_code"
        lost = lost[lost[col] == place["code"]]
    if lost.empty:
        return {"text": "ไม่พบรถที่สัญญาณหายระหว่างวิ่งตามเงื่อนไขนี้"}
    out = []
    for r in lost.sort_values("gps_time", ascending=False).head(3).itertuples():
        hrs = (when[1] - r.gps_time).total_seconds() / 3600
        km = min(r.speed * hrs, 150)                                 # ไม่เดาไกลเกิน 150 กม.
        rad = np.radians(r.direction)
        lat2 = r.lat + km / 111 * np.cos(rad)
        lon2 = r.lon + km / (111 * np.cos(np.radians(r.lat))) * np.sin(rad)
        i = _dist_idx(lat2, lon2)
        if i < 0:
            i = int(hav_km(lat2, lon2, _dist_cent.lat, _dist_cent.lon).idxmin())
        out.append(f"{r.plate} สัญญาณหาย {fmt_t(r.gps_time)} ที่ {r.dist_th}, {r.prov_th} (วิ่ง {r.speed:.0f} กม./ชม. ทิศ {r.direction:.0f}°) "
                   f"→ ถ้าวิ่งต่อทางเดิม น่าจะไปทาง {dist.th.iloc[i]}, {dist.pth.iloc[i]}")
    return {"text": " · ".join(out) + f" (พบรถสัญญาณหายระหว่างวิ่ง {len(lost)} คัน · เป็นการคาดเดาจากทิศทาง ไม่ใช่ตำแหน่งจริง · ข้อมูลไม่มีชื่อถนน)"}


def x_top_trips(when=None):
    s = pos.sort_values(["car_id", "gps_time"])
    f = s.meter.eq("F")
    start = f & ~f.groupby(s.car_id).shift(fill_value=False)        # เปลี่ยนจากว่าง → มีผู้โดยสาร = รับ 1 ครั้ง
    dt = s.groupby("car_id").gps_time.diff().shift(-1)
    dur = dt.where(f & (dt <= STALE), pd.Timedelta(0))
    g = pd.DataFrame({"car_id": s.car_id, "plate": s.plate, "trip": start, "dur": dur}) \
          .groupby(["car_id", "plate"]).agg(trips=("trip", "sum"), dur=("dur", "sum")).reset_index()
    top = g.sort_values(["trips", "dur"], ascending=False).head(3)
    lst = " · ".join(f"{r.plate} รับ {int(r.trips)} ครั้ง รวม {r.dur.total_seconds() / 3600:.1f} ชั่วโมง" for r in top.itertuples())
    return {"text": f"รถที่รับผู้โดยสารบ่อยสุด ({th_date(pos.gps_time.min())} – {th_date(LATEST)}): {lst} "
                    f"· เฉลี่ยทั้งฝูง {g.trips.mean():.1f} ครั้งต่อคัน (นับจากมิเตอร์เปลี่ยนเป็นมีผู้โดยสาร)"}


def x_status_ratio(place, when):
    now = latest_at(when[1])
    scope = "ทั้งหมด"
    if place:
        col = "prov_code" if place["level"] == "prov" else "dist_code"
        now, scope = now[now[col] == place["code"]], place["label"]
    vc = now.status.value_counts()
    pct = (vc / vc.sum() * 100).round().astype(int)
    hired, park, run = pct.get("มีผู้โดยสาร", 0), pct.get("จอด", 0), pct.get("ว่าง·วิ่ง", 0)
    def ratio(a, b):                                                # ปรับให้รวมเป็น 100 เช่น 11:58 → 16:84
        x = round(a / (a + b) * 100) if a + b else 0
        return f"{x}:{100 - x}"
    return {"text": f"ณ {fmt_t(when[1])} รถออนไลน์{scope} {vc.sum():,} คัน · มีผู้โดยสาร {hired}% · วิ่งว่าง {run}% · จอด {park}% "
                    f"→ รถวิ่ง : รถจอด ≈ {ratio(hired + run, park)} · รถมีผู้โดยสาร : รถจอด ≈ {ratio(hired, park)}"}


# ── ตัวเลือกประเภทคำถาม ──────────────────────────────────────────────────
EXAMPLES = {
    "peak_hour": ["เวลากี่โมงมีผู้โดยสารมากที่สุด", "ช่วงไหนคนเรียกแท็กซี่เยอะสุด", "ชั่วโมงเร่งด่วนคือกี่โมง",
                  "ผู้โดยสารเยอะช่วงเวลาไหน", "ช่วงเวลาที่รถว่างน้อยที่สุด", "peak hour ตอนไหน"],
    "data_range": ["มีข้อมูลของวันไหนบ้าง", "ข้อมูลเริ่มวันไหนถึงวันไหน", "ข้อมูลล่าสุดถึงเมื่อไหร่",
                   "ช่วงวันที่ของข้อมูล", "ข้อมูลมีกี่วัน", "อัปเดตข้อมูลล่าสุดกี่โมง"],
    "chargers_in_area": ["มีสถานีชาร์จกี่แห่งในสถานที่", "สถานที่มีปั๊มอีวีกี่ที่", "จำนวนสถานีชาร์จรถไฟฟ้าในสถานที่",
                         "นับที่ชาร์จ EV ในสถานที่", "ทั้งประเทศมีสถานีชาร์จกี่แห่ง",
                         "มีสถานีชาจรถevกี่แห่งในสถานที่", "สถานที่มีที่ชาร์จ ev กี่จุด", "charger ในสถานที่มีกี่ที่"],
    "cars_near_chargers": ["มีรถจอดอยู่ที่สถานีชาร์จกี่คัน", "รถกำลังชาร์จอยู่กี่คัน", "รถจอดใกล้ปั๊ม EV กี่คัน",
                           "ตอนนี้มีรถอยู่ที่จุดชาร์จกี่คัน", "สถานีชาร์จไหนมีรถจอดเยอะสุด"],
    "site_recommend": ["ถ้าอยากเปิดปั๊ม EV ควรเปิดเขตไหน", "ควรตั้งสถานีชาร์จที่ไหนดี", "ทำเลไหนเหมาะเปิดจุดชาร์จ",
                       "พื้นที่ไหนขาดสถานีชาร์จ", "เปิดปั๊มอีวีที่อำเภอไหนดีในสถานที่", "เขตไหนสถานีชาร์จไม่พอ"],
    "lost_signal": ["รถสัญญาณหายคิดว่าวิ่งไปไหน", "รถที่โดนตัดสัญญาณน่าจะไปที่ไหน", "สัญญาณขาดล่าสุดที่ไหน รถจะไปไหนต่อ",
                    "รถที่ขาดการติดต่อไปทางไหน", "รถมีสัญญาณสุดท้ายที่ถนนสุขุมวิท คิดว่าวิ่งไปไหน", "เดาปลายทางรถที่สัญญาณหาย"],
    "top_trips": ["รถคันไหนรับผู้โดยสารบ่อยสุด", "คันไหนวิ่งงานเยอะที่สุด", "รถที่รับลูกค้ามากที่สุด",
                  "คนขับคันไหนขยันสุด", "อันดับรถที่มีเที่ยวผู้โดยสารมากสุด"],
    "status_ratio": ["เทียบสัดส่วนรถที่วิ่งกับรถที่จอด", "รถจอดกี่เปอร์เซ็นต์", "สัดส่วนรถมีผู้โดยสารต่อรถว่าง",
                     "อัตราส่วนรถวิ่งต่อรถจอด", "รถมีผู้โดยสารคิดเป็นกี่เปอร์เซ็นต์"],
    # คำถามที่หน้าตาคล้าย แต่ระบบเดิม (ส่วน 6) ตอบอยู่แล้ว → ส่งกลับ
    "_pass": ["แถวสถานที่มีปั๊มชาร์จไหม", "ที่ชาร์จใกล้สถานที่", "หาที่ชาร์จแถวสถานที่", "สถานีชาร์จใกล้รถ ทะเบียนรถ",
              "สถานที่มีรถกี่คัน", "รถออกนอกเขตกี่คัน", "ทะเบียนรถ อยู่ไหน",
              "แถวสถานที่มีแท็กซี่วิ่งอยู่ไหมตอนนี้", "ตอนนี้สถานที่มีรถวิ่งไหม", "สถานที่มีรถว่างไหม", "รถออกนอกเขตวันนี้ไปไหนบ้าง"],
}
_ex_q = [(k, q) for k, qs in EXAMPLES.items() for q in qs]
_ex_v = enc.encode([mask(q) for _, q in _ex_q], normalize_embeddings=True)
_base = train_df.groupby("intent").head(40)                          # ตัวอย่างคำถามแบบเดิม (ส่วน 4) ไว้เทียบ
_base_v = enc.encode(_base.masked.tolist(), normalize_embeddings=True)

ANALYTIC = {
    "peak_hour": lambda e: x_peak_hour(),
    "data_range": lambda e: x_data_range(),
    "chargers_in_area": lambda e: x_chargers_in_area(e["place"]),
    "cars_near_chargers": lambda e: x_cars_near_chargers(e["place"], e["when"]),
    "site_recommend": lambda e: x_site_recommend(e["place"]),
    "lost_signal": lambda e: x_lost_signal(e["plate"], e["place"], e["when"]),
    "top_trips": lambda e: x_top_trips(),
    "status_ratio": lambda e: x_status_ratio(e["place"], e["when"]),
}


def route_analytic(q, verbose=False):
    """คืน (intent, ข้อความข้อเท็จจริง) ถ้าเป็นคำถามวิเคราะห์ · ไม่ใช่ = None"""
    v = enc.encode([mask(q)], normalize_embeddings=True)[0]
    sim = _ex_v @ v
    i = int(sim.argmax())
    intent, score = _ex_q[i][0], float(sim[i])
    base = float((_base_v @ v).max())                                # ใกล้คำถามแบบเดิมมากกว่า → ให้ระบบเดิมตอบ
    if verbose:
        print(f"  analytic? {intent} ({score:.2f} · แบบเดิม {base:.2f}) ใกล้ตัวอย่าง: {_ex_q[i][1]}")
    if score < ROUTE_MIN or score < base or intent == "_pass":
        return None
    return intent, ANALYTIC[intent](extract(q))["text"]


# ห่อ ask() ส่วน 7 (รันซ้ำได้ ไม่ห่อซ้อน)
if "_ask_pre_analytic" not in globals():
    _ask_pre_analytic = ask
def ask(q, verbose=False):
    r = route_analytic(q, verbose)
    return r[1] if r else _ask_pre_analytic(q, verbose)


for q in ["เวลาประมาณกี่โมงมีผู้โดยสารมากที่สุด", "มีข้อมูลของวันไหนบ้าง", "มีสถานีชาจรถevกี่แห่งในกรุงเทพ",
          "มีรถจอดอยู่ที่สถานีจอดรถevกี่คัน", "ถ้าอยากเปิดปั้ม ev ควรเปิดเขตไหน",
          "รถมีสัญญาณที่ถนนสุขุมวิทสุดท้ายก่อนโดนตัด คิดว่าวิ่งไปไหน", "รถคันไหนที่รับผู้โดยสารบ่อยสุด",
          "เทียบสัดส่วนรถที่วิ่งกับรถที่จอด", "ตอนนี้ภูเก็ตมีรถกี่คัน"]:
    print("Q:", q); print("A:", ask(q, verbose=True)); print()

## 8 · (ทางเลือก) ใช้ LLM รันเอง + สอนเพิ่ม (fine-tune)

### ทำไมต้องมีส่วนนี้
ส่วน 1–7 ตอบได้เฉพาะ 8 แบบที่สอนไว้ · ถ้าผู้ใช้ถามยาก ๆ เช่น *"รถที่อยู่ชลบุรีแต่เมื่อวานอยู่กรุงเทพมีกี่คัน"* ตัวจัดประเภทจะทำไม่ได้ · **LLM** (Large Language Model — โมเดลภาษาขนาดใหญ่แบบ ChatGPT) เข้าใจประโยคซับซ้อนได้ดีกว่า

### คำศัพท์ที่ต้องรู้ก่อน
| คำ | ความหมายแบบง่าย |
|---|---|
| **LLM โอเพนซอร์ส** | LLM ที่ดาวน์โหลดตัวโมเดลมารันเองได้ (ไม่ต้องส่งข้อมูลไปบริษัทอื่น) เช่น Qwen · Typhoon |
| **fine-tune** | สอนโมเดลที่ฉลาดอยู่แล้วให้ทำงานเฉพาะของเรา ด้วยตัวอย่างไม่กี่พันข้อ |
| **LoRA** | วิธี fine-tune แบบประหยัด: ไม่แก้โมเดลทั้งตัว (พันล้านตัวเลข) แต่เพิ่ม "ชิ้นส่วนเสริม" เล็ก ๆ (~1% ของโมเดล) แล้วเทรนแค่ชิ้นนั้น → ใช้หน่วยความจำน้อย ไฟล์ผลลัพธ์เล็ก (หลักสิบ MB) |
| **4-bit (quantization)** | บีบโมเดลให้เล็กลง ~4 เท่า (เก็บตัวเลขด้วย 4 บิตแทน 16) → ใส่ GPU ฟรีของ Colab ได้ · ความแม่นลดลงเล็กน้อย |
| **GPU / T4** | ชิปที่คำนวณงาน AI เร็วกว่า CPU หลายสิบเท่า · Colab ฟรีให้ T4 (หน่วยความจำ 15 GB) ใช้ได้จำกัดชั่วโมง |
| **token** | ชิ้นส่วนของข้อความที่โมเดลอ่าน (ประมาณคำ/พยางค์) |
| **loss** | ตัวเลขบอกว่าโมเดลยังทายผิดมากแค่ไหน — ระหว่างเทรนควร**ลดลง** |

### เราสอน LLM ให้ทำอะไร
**ไม่ได้สอนให้จำตำแหน่งรถ** (ข้อมูลเปลี่ยนทุกนาที สอนไปก็ล้าสมัย) · สอนให้ **แปลงคำถาม → คำสั่ง JSON** แล้วให้ executor ส่วน 6 ไปค้นข้อมูลจริงเหมือนเดิม

```
คำถาม: ภูเกตตอนนี้มีรถกี่คัน
JSON:  {"intent": "cars_in_area", "plate": null, "place": "ภูเก็ต", "time": "ล่าสุด"}
```
→ LLM ทำหน้าที่แทนส่วน 3 + 5 (เข้าใจคำถาม) · ส่วน 6 + 7 (ค้นข้อมูล + ตอบ) ใช้ของเดิม

### ขั้นตอนทั้งหมด (แต่ละขั้นคือ 1 cell ข้างล่าง)
| ขั้น | ทำอะไร | เวลาโดยประมาณ (T4) |
|---|---|---|
| 8.1 | เปลี่ยนเครื่องเป็น GPU + ตรวจ | 1 นาที |
| 8.2 | เตรียมตัวอย่างสอน (คำถาม → JSON) | ไม่กี่วินาที |
| 8.3 | ติดตั้งไลบรารี | 1–2 นาที |
| 8.4 | ดาวน์โหลด + โหลดโมเดลแบบ 4-bit | 2–5 นาที |
| 8.5 | ติด LoRA | ไม่กี่วินาที |
| 8.6 | แปลงตัวอย่างเป็น token | ไม่กี่วินาที |
| 8.7 | เทรน | 10–20 นาที |
| 8.8 | ทดสอบ เทียบกับตัวจัดประเภทส่วน 5 | 1–2 นาที |
| 8.9 | บันทึกลง Drive + วิธีโหลดกลับมาใช้ | ไม่กี่วินาที |
| 8.10 | ถามด้วย LLM ผ่าน `ask_llm()` | — |

เวลาเป็นค่าประมาณ `[อนุมาน — ยังไม่ได้ทดสอบบน GPU จริง]`

### 8.1 · เปิด GPU และเปิดส่วนนี้

1. แผงขวา **Settings → Accelerator → GPU T4 x2** (ถ้าเปลี่ยนตอนนี้ เครื่องจะเริ่มใหม่ → รันส่วน 0–7 ใหม่)
2. แก้ `RUN_LLM = False` เป็น `RUN_LLM = True` ใน cell ข้างล่าง แล้วรัน
3. ผลที่ถูก: `GPU: Tesla T4 ...` · ถ้าขึ้นไม่มี GPU → ตรวจ Accelerator หรือโควตา GPU สัปดาห์นี้หมด (ดูที่ kaggle.com → Settings)

**เลือกโมเดล (`LLM_ID`)** — ค่าเริ่มต้น `Qwen/Qwen2.5-1.5B-Instruct` · ต้องเป็นรุ่นเดียวกับที่เทรน `lora_taxi` ไว้

In [ ]:
RUN_LLM = False          # ← เปลี่ยนเป็น True หลังทำขั้น 1–3 ข้างบน
LLM_ID = "Qwen/Qwen2.5-1.5B-Instruct"

if RUN_LLM:
    assert torch.cuda.is_available(), "ไม่มี GPU — Settings → Accelerator → GPU T4 x2 แล้วรันส่วน 0–7 ใหม่"
    print("GPU:", torch.cuda.get_device_name(0), f"· หน่วยความจำ {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
else:
    print("ข้ามส่วน 8 (RUN_LLM = False) · GPU:", torch.cuda.is_available())

### 8.2 · เตรียมตัวอย่างสอน (คำถาม → JSON)

ใช้ประโยคจากส่วน 4 (`train_df`) มาสร้างคู่ **โจทย์ (prompt)** กับ **เฉลย (completion)** 2,000 คู่
- prompt = `แปลงคำถามเป็นคำสั่ง JSON / คำถาม: ... / JSON:`
- completion = JSON ที่ถูกต้อง — สร้างจาก intent ที่รู้อยู่แล้ว + slot ที่ส่วน 3 ดึงได้

cell นี้รันได้แม้ไม่มี GPU — ดูตัวอย่าง 3 คู่ว่าหน้าตาถูกต้องก่อนเทรน (**ถ้าเฉลยผิด โมเดลก็จะเรียนผิด**)

In [ ]:
def to_target(q, intent):
    e = extract(q)
    return json.dumps({"intent": intent, "plate": e["plate"], "place": e["place"] and e["place"]["label"],
                       "time": e["when"][2]}, ensure_ascii=False)

PROMPT = "แปลงคำถามเป็นคำสั่ง JSON\nคำถาม: {q}\nJSON:"
sft = [{"prompt": PROMPT.format(q=r.q), "completion": " " + to_target(r.q, r.intent)}
       for r in train_df.sample(min(2000, len(train_df)), random_state=0).itertuples()]
print(len(sft), "คู่")
for x in sft[:3]:
    print("-" * 60); print(x["prompt"] + x["completion"])

### 8.3 · ติดตั้งไลบรารี

Colab มี `transformers` และ `torch` อยู่แล้ว · ต้องติดตั้งเพิ่ม 3 ตัว:
- `peft` — ทำ LoRA
- `bitsandbytes` — บีบโมเดลเป็น 4-bit
- `accelerate` — จัดการวางโมเดลลง GPU

ถ้าขึ้นข้อความแนะนำให้ **Restart session** หลังติดตั้ง → กด Restart แล้วรันส่วน 0–7 + 8.1–8.2 ใหม่

In [ ]:
if RUN_LLM:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "bitsandbytes", "accelerate"], check=True)
    import peft, bitsandbytes
    print("peft", peft.__version__, "· bitsandbytes", bitsandbytes.__version__)
else:
    print("⏭ ข้ามขั้นนี้ — ยังเป็น RUN_LLM = False (แก้ใน cell 8.1 แล้วรัน 8.1 ใหม่)")

### 8.4 · ดาวน์โหลดและโหลดโมเดลแบบ 4-bit

- ครั้งแรกจะดาวน์โหลดโมเดลจาก Hugging Face (~3 GB สำหรับ 1.5B) — รอได้
- `BitsAndBytesConfig(load_in_4bit=True)` = บีบให้เล็กลงตอนโหลด
- `float16` เพราะ T4 ไม่รองรับ `bfloat16`
- **tokenizer** = ตัวตัดข้อความเป็น token ของโมเดลรุ่นนี้ (ต้องใช้คู่กับโมเดลเดียวกันเสมอ)

ผลที่ถูก: แสดงหน่วยความจำ GPU ที่ใช้ (~1.5–2.5 GB)

In [ ]:
if RUN_LLM:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    tok = AutoTokenizer.from_pretrained(LLM_ID)
    tok.pad_token = tok.pad_token or tok.eos_token          # ตัวเติมช่องว่างให้ประโยคยาวเท่ากันในชุดเดียว
    model = AutoModelForCausalLM.from_pretrained(
        LLM_ID, device_map="auto",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                               bnb_4bit_compute_dtype=torch.float16))
    print(f"โหลดแล้ว · ใช้หน่วยความจำ GPU {torch.cuda.memory_allocated() / 1e9:.1f} GB")
else:
    print("⏭ ข้ามขั้นนี้ — ยังเป็น RUN_LLM = False (แก้ใน cell 8.1 แล้วรัน 8.1 ใหม่)")

### 8.5 · ติด LoRA

- `r=16` = ขนาดของชิ้นส่วนเสริม (ใหญ่ขึ้น = เรียนได้มากขึ้นแต่ช้าลง · 8–32 เป็นค่าทั่วไป)
- `target_modules` = ส่วนของโมเดลที่ติดชิ้นเสริม (ส่วน attention ของ Qwen/Llama ใช้ชื่อ `q_proj k_proj v_proj o_proj` — โมเดลตระกูลอื่นอาจใช้ชื่ออื่น ถ้า error เรื่องชื่อ ให้ดูชื่อจาก `print(model)`)

ผลที่ถูก: `trainable params: ... || all params: ... || trainable%: ~0.5–1` = เทรนแค่ ~1% ของโมเดล

In [ ]:
if RUN_LLM:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    model = prepare_model_for_kbit_training(model)           # เตรียมโมเดล 4-bit ให้เทรนได้
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
    model.print_trainable_parameters()
else:
    print("⏭ ข้ามขั้นนี้ — ยังเป็น RUN_LLM = False (แก้ใน cell 8.1 แล้วรัน 8.1 ใหม่)")

### 8.6 · แปลงตัวอย่างเป็น token

โมเดลเรียนจากตัวเลข จึงต้องแปลงข้อความเป็น token id
**จุดสำคัญ:** ใส่ label `-100` ให้ส่วนโจทย์ = บอกโมเดลว่า "ไม่ต้องเรียนการพิมพ์โจทย์ ให้เรียนแค่การตอบ JSON"
ท้ายเฉลยเติม `eos_token` (สัญลักษณ์จบข้อความ) ให้โมเดลรู้ว่าตอบจบแล้ว ไม่พิมพ์ต่อเรื่อย ๆ

In [ ]:
if RUN_LLM:
    def encode_ex(ex, max_len=256):
        p = tok(ex["prompt"], add_special_tokens=False)["input_ids"]
        c = tok(ex["completion"] + tok.eos_token, add_special_tokens=False)["input_ids"]
        ids = (p + c)[:max_len]
        return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": ([-100] * len(p) + c)[:max_len]}

    train_ds = [encode_ex(x) for x in sft]
    x = train_ds[0]
    print("token ทั้งหมด", len(x["input_ids"]), "· ส่วนที่เรียน (ไม่ใช่ -100)", sum(l != -100 for l in x["labels"]))
else:
    print("⏭ ข้ามขั้นนี้ — ยังเป็น RUN_LLM = False (แก้ใน cell 8.1 แล้วรัน 8.1 ใหม่)")

### 8.7 · เทรน

| ค่า | ความหมาย |
|---|---|
| `per_device_train_batch_size=8` | ป้อนทีละ 8 ตัวอย่าง (ถ้า error `CUDA out of memory` ลดเป็น 4) |
| `gradient_accumulation_steps=2` | สะสม 2 รอบก่อนปรับโมเดล = เหมือน batch 16 |
| `num_train_epochs=1` | ดูตัวอย่างทั้งหมด 1 รอบ (2,000 ข้อ ≈ 125 รอบการปรับ) |
| `learning_rate=2e-4` | ขนาดก้าวในการปรับ (ค่ามาตรฐานของ LoRA) |
| `fp16=True` | คำนวณด้วยความละเอียดครึ่งหนึ่ง เร็วขึ้นบน T4 |

**ระหว่างเทรน** ตาราง `Step / Training Loss` จะขึ้นทุก 20 รอบ — **loss ควรลดลง** (เช่น จาก ~2 ลงไปต่ำกว่า 0.2) · ถ้าไม่ลดเลย = ตั้งค่าผิดหรือข้อมูลผิด
⚠️ อย่าปิดแท็บ / ปล่อยว่างนาน — Colab ฟรีตัดการเชื่อมต่อเมื่อไม่มีการใช้งาน (ทุกอย่างหาย ต้องเริ่มใหม่)

In [ ]:
if RUN_LLM:
    from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
    trainer = Trainer(
        model=model, train_dataset=train_ds,
        data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
        args=TrainingArguments(output_dir="/kaggle/working/lora_ckpt", per_device_train_batch_size=8,
                               gradient_accumulation_steps=2, num_train_epochs=1, learning_rate=2e-4,
                               fp16=True, logging_steps=20, save_strategy="no", report_to=[]))
    out = trainer.train()
    m = out.metrics
    print(f"เทรนเสร็จ {m['train_runtime'] / 60:.1f} นาที · {out.global_step} รอบการปรับ · loss เฉลี่ย {m['train_loss']:.3f}")
    if m["train_runtime"] < 60:
        print("⚠️ เร็วผิดปกติ — ตรวจว่า train_ds มีข้อมูล (len(train_ds)) และรันขั้น 8.2–8.6 ครบ")
else:
    print("⏭ ข้ามขั้นนี้ — ยังเป็น RUN_LLM = False (แก้ใน cell 8.1 แล้วรัน 8.1 ใหม่)")

### 8.8 · ทดสอบ — เทียบกับตัวจัดประเภทส่วน 5

`llm_parse(คำถาม)` ให้โมเดลเขียน JSON (`do_sample=False` = ตอบแบบเดิมทุกครั้ง ไม่สุ่ม) แล้วแปลงเป็น dict · ถ้าโมเดลเขียน JSON ผิดรูปแบบ จะคืน `intent: None` (ระบบจริงต้องถามกลับ)
ทดสอบกับคำถามที่เขียนเอง 20 ข้อ (`hand`) — คอลัมน์ `pred` = ตัวจัดประเภทส่วน 5 · `llm` = LLM

**ตัดสินใจยังไง:** ถ้า LLM ไม่ได้ถูกมากกว่าชัดเจน → ใช้ตัวจัดประเภทส่วน 5 ต่อ (เร็วกว่า ไม่ต้องใช้ GPU) · LLM คุ้มเมื่อมีคำถามซับซ้อนที่ส่วน 5 ทำไม่ได้ — ควรเพิ่มคำถามแบบนั้นลง `HAND` เพื่อวัดด้วย

In [ ]:
if RUN_LLM:
    model.eval()

    def llm_parse(q):
        x = tok(PROMPT.format(q=q), return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**x, max_new_tokens=80, do_sample=False, pad_token_id=tok.pad_token_id)
        txt = tok.decode(out[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        try:
            return json.loads(txt.splitlines()[0])
        except Exception:
            return {"intent": None, "raw": txt}

    got = [llm_parse(q) for q in hand.q]
    hand["llm"] = [g.get("intent") for g in got]
    print(f"LLM ถูก {(hand.llm == hand.intent).mean():.0%} · ตัวจัดประเภทส่วน 5 ถูก {(hand.pred == hand.intent).mean():.0%}")
    display(hand[["q", "intent", "pred", "llm"]])
else:
    print("⏭ ข้ามขั้นนี้ — ยังเป็น RUN_LLM = False (แก้ใน cell 8.1 แล้วรัน 8.1 ใหม่)")

### 8.9 · บันทึก LoRA (และวิธีโหลดกลับมาใช้)

ถ้าเทรนใหม่ใน Kaggle: cell ข้างล่างบันทึกลง `/kaggle/working/lora_taxi/` → หลังรันเสร็จ ดาวน์โหลดจากแท็บ **Output** แล้วอัปโหลดเข้า dataset เพื่อใช้ครั้งหน้า

**ใช้ตัวที่เทรนไว้แล้ว (ไม่ต้องเทรนใหม่):** รัน 8.1 → 8.2 → 8.3 → **8.9b** (ข้าม 8.4–8.7) → 8.8 → 8.10

In [ ]:
if RUN_LLM and CLOUD:
    model.save_pretrained(str(DRIVE / "lora_taxi"))
    tok.save_pretrained(str(DRIVE / "lora_taxi"))
    print("บันทึกแล้ว →", DRIVE / "lora_taxi")
else:
    print("⏭ ข้ามขั้นนี้ — ยังเป็น RUN_LLM = False (แก้ใน cell 8.1 แล้วรัน 8.1 ใหม่)")

In [ ]:
# ── 8.9b · โหลดโมเดล LoRA ที่เทรนไว้แล้วจาก dataset (ไม่ต้องเทรนใหม่) ──
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

assert RUN_LLM, "แก้ RUN_LLM = True ใน cell 8.1 แล้วรัน 8.1 ใหม่ก่อน"
assert LORA_IN, "ไม่พบโฟลเดอร์ lora_taxi ใน /kaggle/input — อัปโหลดเข้า dataset แล้ว Add Input"

tok = AutoTokenizer.from_pretrained(LLM_ID)
base = AutoModelForCausalLM.from_pretrained(
    LLM_ID, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16))
model = PeftModel.from_pretrained(base, str(LORA_IN))
model.eval()
print("โหลดโมเดล LoRA แล้ว:", LORA_IN)

### 8.10 · ถามด้วย LLM

`ask_llm()` ทำงานเหมือน `ask()` ส่วน 7 แต่ใช้ LLM เข้าใจคำถามแทนส่วน 3 + 5
- LLM บอกชื่อสถานที่เป็นข้อความ → ส่งต่อให้ `find_place()` แปลงเป็นรหัสอีกที (กันชื่อที่ LLM เขียนไม่ตรง)
- ทะเบียนและเวลา ใช้ตัวดึงของส่วน 3 ตามเดิม (แม่นกว่าและตรวจสอบได้)
- ค้นข้อมูลและตอบ ใช้ส่วน 6 + 7 เหมือนเดิม → **ตัวเลขมาจากข้อมูลจริงเสมอ**

In [ ]:
if RUN_LLM:
    def ask_llm(q):
        j = llm_parse(q)
        intent = j.get("intent")
        if intent not in set(T):
            return f"ไม่เข้าใจคำถาม (LLM ตอบ: {j.get('raw', j)})"
        if intent == "action_request":
            return "ระบบนี้ตอบคำถามได้อย่างเดียว ไม่สามารถสั่งการรถ (ตัดสัญญาณ/ล็อก/ดับเครื่อง) ได้"
        if intent == "other":
            return "ตอบได้เฉพาะคำถามเรื่องรถแท็กซี่ EV7"
        e = extract(q)                                             # ทะเบียน/เวลา จากตัวดึงส่วน 3
        place = find_place(j["place"]) if j.get("place") else e["place"]
        run = {"cars_in_area": lambda: x_cars_in_area(place, e["when"]),
               "car_status": lambda: x_car_status(e["plate"], e["when"]),
               "car_track": lambda: x_car_track(e["plate"], e["when"]),
               "out_of_zone": lambda: x_out_of_zone(e["when"]),
               "fleet_stats": lambda: x_fleet_stats(e["when"]),
               "nearest_charger": lambda: x_nearest_charger(e["plate"], place, e["when"])}[intent]
        return answer(intent, run())

    for q in ["ตอนนี้รถคันไหนอยู่ภูเกตบ้าง", f"{demo_plate} อยู่ไหน", "ช่วยดูหน่อยว่ารถที่อยู่โคราชตอนนี้มีกี่คัน"]:
        print("Q:", q); print("A:", ask_llm(q)); print()
else:
    print("⏭ ข้ามขั้นนี้ — ยังเป็น RUN_LLM = False (แก้ใน cell 8.1 แล้วรัน 8.1 ใหม่)")

### 8.11 · ลองถามเอง — ดูว่าระบบเข้าใจคำถามยังไง

`ลองถาม("คำถาม", ...)` แสดงทีละคำถาม:
1. **ค่าที่ดึงได้ (slot)** — ทะเบียน · สถานที่ (พร้อมคะแนนความเหมือน) · เวลา
2. **วิธีส่วน 5 (ตัวจัดประเภท)** — intent ที่ทาย + ความมั่นใจ + intent อันดับ 2–3 + คำตอบ
3. **วิธีส่วน 8 (LLM)** — JSON ที่ LLM เขียน + คำตอบ (แสดงเฉพาะเมื่อเทรน LLM แล้ว)

ใช้ได้ทั้งตอน `RUN_LLM = False` (เห็นแค่ข้อ 1–2) และ `True`

**ถามแบบพิมพ์สด:** รัน `ถามสด()` แล้วพิมพ์คำถามในช่องที่ขึ้นมา · กด Enter ว่าง ๆ เพื่อจบ

In [ ]:
def ลองถาม(*questions):
    llm_ready = "llm_parse" in globals()
    for q in questions:
        print("═" * 90); print("Q:", q)
        e = extract(q)
        pl = e["place"]
        place_txt = f"{pl['label']} ({pl['score']})" if pl else None
        print(f"  slot  ทะเบียน={e['plate']} · สถานที่={place_txt} · เวลา={e['when'][2]}")
        p = clf.predict_proba(enc.encode([mask(q)], normalize_embeddings=True))[0]
        top = np.argsort(p)[::-1][:3]
        print("  [ส่วน 5] intent:", " · ".join(f"{clf.classes_[k]} {p[k]:.2f}" for k in top))
        print("  [ส่วน 5] ตอบ:", ask(q))
        if llm_ready:
            print("  [LLM]  JSON:", llm_parse(q))
            print("  [LLM]  ตอบ:", ask_llm(q))
        else:
            print("  [LLM]  — ยังไม่ได้เทรน (RUN_LLM = False)")

def ถามสด():
    while True:
        q = input("ถาม (Enter ว่าง = จบ): ").strip()
        if not q:
            break
        ลองถาม(q)

ลองถาม("ตอนนี้รถคันไหนอยู่ภูเกตบ้าง",
       f"{demo_plate} อยู่ไหน",
       "รถที่อยู่ชลบุรีมีกี่คัน",
       f"เมื่อวาน {demo_plate} ไปไหนมาบ้าง",
       "ใกล้ๆหาดใหญ่มีที่ชาร์จไหม",
       "อยากรู้ว่ารถคันไหนวิ่งไปต่างจังหวัดบ้างวันนี้")

## 9 · ขั้นต่อไป — จาก notebook สู่การใช้งานจริง

### 9.1 เก็บคำถามจริงจากผู้ใช้ (สำคัญที่สุด)
ตอนนี้โมเดลเรียนจากประโยคที่เราแต่งเอง · ผู้ใช้จริงพิมพ์ไม่เหมือนที่เราคิดเสมอ
1. ให้ผู้ใช้ทดลองถาม (หรือเปิด Google Form ให้พิมพ์ "อยากถามอะไรระบบนี้") — เป้าหมายแรก **50–100 ข้อ**
2. บันทึกเป็นตาราง 2 คอลัมน์: `q` (คำถาม) · `intent` (คำตอบที่ถูก 1 ใน 8 แบบ) เช่น

   | q | intent |
   |---|---|
   | ภูเก็ตตอนนี้มีรถเรามั้ย | cars_in_area |
   | EV-0042 ตอนนี้ไปถึงไหนแล้ว | car_status |

3. **ข้อที่ไม่เข้า 8 แบบ** → จดแยกไว้ ถ้าเจอซ้ำบ่อยค่อยเพิ่ม intent ใหม่ (ต้องเขียน executor ส่วน 6 เพิ่มด้วย)

### 9.2 วงจรปรับปรุง (ทำซ้ำทุกครั้งที่มีคำถามใหม่)
```
คำถามจริง ─► เพิ่มเป็นชุดทดสอบ (HAND) ─► รันส่วน 5 ดูข้อที่ผิด
     ▲                                          │
     └── เทรนใหม่ ◄── เพิ่ม template ที่คล้ายข้อที่ผิด (ส่วน 4) ◄┘
```
อย่าเอาคำถามชุดทดสอบไปใส่ในชุดเทรนตรง ๆ — คะแนนจะสูงเกินจริง

### 9.3 เลือกว่าจะใช้วิธีไหนในระบบจริง
| ถ้า | ใช้ |
|---|---|
| คำถามส่วนใหญ่อยู่ใน 8 แบบ และส่วน 5 ถูก ≥ 90% | **ตัวจัดประเภทส่วน 5** — เร็ว ไม่ต้องใช้ GPU ข้อมูลไม่ออกนอกองค์กร |
| คำถามซับซ้อนเยอะ และ LLM (ส่วน 8) ถูกกว่าชัดเจน | **LLM รันเอง** — ต้องมีเครื่อง GPU ถาวร (Colab ฟรีใช้ทำ production ไม่ได้ ตัดการเชื่อมต่อบ่อย) |
| ไม่อยากดูแลเครื่อง GPU และ DPO อนุมัติให้ส่งข้อมูลออก | **API** (ดูโน้ต Q&A Assistant Design) |

### 9.4 ทำเป็นบริการให้หน้าแผนที่เรียกใช้
1. ย้ายโค้ดส่วน 1–7 เป็นไฟล์ `.py` บนเครื่องที่ข้อมูลอยู่ (ไม่ใช่ Colab)
2. เปิดเป็น API เล็ก ๆ ด้วย FastAPI — ตัวอย่างโครง:
   ```python
   from fastapi import FastAPI
   app = FastAPI()

   @app.get("/ask")
   def ask_api(q: str):
       return {"answer": ask(q)}
   ```
   รันด้วย `uvicorn app:app` → เรียก `http://<server>:8000/ask?q=...`
3. ให้ API คืน **รายการ car_id** ด้วย → หน้าแผนที่เอาไปไฮไลต์รถ และแปลงรหัสเป็นทะเบียนจริงฝั่งเว็บ (ทะเบียนจริงไม่ต้องผ่านโมเดล)
4. ต่อข้อมูลจริงแทนไฟล์ parquet เมื่อทำ realtime (Phase 6 ของแผนโปรเจกต์)

### 9.5 ความปลอดภัย
- ระบบนี้ **อ่านอย่างเดียว** · ไม่มีคำสั่งใดที่ควบคุมรถได้ (คำขอแบบนั้นถูกจัดเป็น `action_request` และปฏิเสธ)
- ล้าง output ก่อนส่ง notebook ต่อ (**Edit → Clear all outputs**)
- ใช้กับข้อมูลจริง (ทะเบียนจริง) เฉพาะบนเครื่ององค์กร · Colab ใช้กับชุดที่แทนทะเบียนแล้วเท่านั้น

## 10 · Chatbot ② Qwen — ฟรี · ตอบเป็นภาษาพูด

ต้องรันส่วน 0–7 (+ 7.1) ก่อน · ใช้ GPU · ครั้งแรกดาวน์โหลดโมเดล 3B (~6 GB) 2–3 นาที
ตัวเลขทุกตัวยังมาจาก executor ส่วน 6 — LLM แค่เรียบเรียงภาษา · ถ้า LLM ใส่ตัวเลขที่ไม่มีในข้อมูล จะใช้คำตอบจากระบบแทน

ลองถาม: `chat("เขตไหนในกรุงเทพมีรถเยอะสุด", show_facts=True)` · คุยต่อเนื่อง: `แชต()`

In [ ]:
# ── 10 · ตอบแบบ chatbot — ภาษาพูด + จำบทสนทนา ─────────────────────────────
# ต้องรันส่วน 0–7 ก่อน · ฟรี · ใช้ GPU T4 (ไม่มี GPU ได้แต่ช้ามาก) · ตัวเลขทุกตัวยังมาจาก executor ส่วน 6 — LLM แค่เรียบเรียงภาษา
import re, random, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

CHAT_LLM = "Qwen/Qwen2.5-3B-Instruct"   # เก่งกว่า: "Qwen/Qwen2.5-7B-Instruct" (โหลดแบบ 4-bit อัตโนมัติ · ~5 GB · ช้าลง ~2 เท่า)
USE_LLM = True                          # False = ไม่ใช้ LLM (ตอบจากแม่แบบที่เป็นกันเองขึ้น)
# เปลี่ยนรุ่นแล้วต้องลบตัวเก่าก่อน:  del cmodel; torch.cuda.empty_cache()  แล้วรัน cell นี้ใหม่

if USE_LLM and "cmodel" not in globals():
    ctok = AutoTokenizer.from_pretrained(CHAT_LLM)
    kw = {}
    if "7B" in CHAT_LLM or "14B" in CHAT_LLM:                # รุ่นใหญ่ บีบเป็น 4-bit ให้พอ GPU T4
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate"], check=True)
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    cmodel = AutoModelForCausalLM.from_pretrained(
        CHAT_LLM, dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None, **kw)
    cmodel.eval()

SYSTEM = ("คุณคือผู้ช่วยของทีม EV7 ตอบคำถามเรื่องรถแท็กซี่ไฟฟ้าเป็นภาษาไทย เหมือนเพื่อนร่วมงานคุยกัน\n"
          "- ตอบเฉพาะสิ่งที่ถาม สั้น 1–2 ประโยค ขึ้นต้นด้วยคำตอบเลย ไม่ทวนคำถาม\n"
          "- ใช้เฉพาะข้อมูลจากระบบ ห้ามเพิ่มตัวเลข ชื่อ หรือทะเบียนที่ไม่มีในข้อมูล\n"
          "- ถ้าข้อมูลมีคำพูดแบบคนไทย (เช่น บ่าย 3) ให้ใช้คำนั้น · ถ้าข้อมูลบอกว่าเป็นการประมาณ/คาดเดา ให้บอกด้วย\n"
          "- ข้อมูลส่วนที่ผู้ใช้ไม่ได้ถาม ไม่ต้องพูดถึง")

# ตัวอย่างคำตอบที่อยากได้ (few-shot) — Qwen จะเลียนแบบความยาวและน้ำเสียง · ตัวเลขในนี้ถูกกันไม่ให้หลุดไปในคำตอบจริงโดย safe()
FEWSHOT = [
    ("ช่วงที่มีผู้โดยสารมากที่สุดคือ 7 โมงเช้า (07:00–07:59) เฉลี่ย 512 คันที่มีผู้โดยสาร · น้อยสุดคือ ตี 4 (98 คัน)",
     "ผู้โดยสารเยอะช่วงไหน", "ช่วง 7 โมงเช้าครับ มีรถรับผู้โดยสารเฉลี่ย 512 คัน"),
    ("ณ 02/10 09:00 รถออนไลน์ทั้งหมด 2,000 คัน · มีผู้โดยสาร 20% · วิ่งว่าง 30% · จอด 50% → รถวิ่ง : รถจอด ≈ 50:50 · รถมีผู้โดยสาร : รถจอด ≈ 29:71",
     "รถมีผู้โดยสารเทียบกับรถจอดเป็นยังไง", "รถมีผู้โดยสาร : รถจอด ประมาณ 29:71 ครับ (ข้อมูล 09:00)"),
    ("เขตที่น่าเปิดสถานีชาร์จในนนทบุรี: 1. อำเภอปากเกร็ด (รถเฉลี่ย 88 คัน/ชม. · สถานีชาร์จ 1 แห่ง) · 2. อำเภอบางใหญ่ (รถเฉลี่ย 61 คัน/ชม. · สถานีชาร์จ 0 แห่ง) — เป็นคำแนะนำเบื้องต้น",
     "นนทบุรีควรเปิดที่ชาร์จตรงไหน", "แนะนำอำเภอปากเกร็ดครับ รถวิ่งเฉลี่ย 88 คันต่อชั่วโมงแต่มีสถานีชาร์จแค่ 1 แห่ง รองลงมาคืออำเภอบางใหญ่ (เป็นการประเมินเบื้องต้น)"),
]

memory = {"plate": None, "place": None}
REF_CAR = re.compile(r"คันนี้|คันเดิม|คันนั้น|คันเมื่อกี้|รถคันนี้|มัน")
REF_PLACE = re.compile(r"ที่นั่น|จังหวัดนี้|แถวนั้น|ที่เดิม|ที่นี่")

def understand(q):
    """ดึง slot + ทาย intent · ถ้าอ้าง 'คันนี้' / 'ที่นั่น' ใช้ค่าจากคำถามก่อนหน้า"""
    e = extract(q)
    extra = ""
    if not e["plate"] and memory["plate"] and REF_CAR.search(q):
        e["plate"] = memory["plate"]; extra += " ทะเบียนรถ"
    if not e["place"] and memory["place"] and REF_PLACE.search(q):
        e["place"] = memory["place"]; extra += " สถานที่"
    p = clf.predict_proba(enc.encode([mask(q) + extra], normalize_embeddings=True))[0]
    return e, clf.classes_[p.argmax()], float(p.max())

def facts_for(q):
    """คืน (intent, ข้อเท็จจริงเป็นข้อความ) — ข้อเท็จจริงมาจาก executor ส่วน 6 / 7.2 เท่านั้น"""
    if "route_analytic" in globals():                      # cell 7.2 — คำถามวิเคราะห์
        r = route_analytic(q)
        if r:
            return r
    if "is_rank" in globals() and is_rank(q):             # cell 7.1 — "เขตไหน…มากที่สุด"
        return "area_ranking", rank_facts(q)
    e, intent, conf = understand(q)
    if conf < MIN_CONF:
        return "unclear", "ระบบไม่แน่ใจว่าผู้ใช้ถามเรื่องอะไร ให้ชวนถามใหม่ เช่น ตำแหน่งรถ รถในจังหวัด เส้นทาง รถนอกเขต จุดชาร์จ"
    if intent == "action_request":
        return intent, "ระบบนี้ตอบคำถามได้อย่างเดียว ไม่สามารถสั่งการรถ (ตัดสัญญาณ ล็อก ดับเครื่อง) ได้"
    if intent == "other":
        return intent, "คำถามนี้ไม่เกี่ยวกับรถแท็กซี่ EV7 ระบบตอบได้เฉพาะ ตำแหน่งรถ รถในพื้นที่ เส้นทาง รถนอกเขต สถานีชาร์จ"
    run = {"cars_in_area": lambda: x_cars_in_area(e["place"], e["when"]),
           "car_status": lambda: x_car_status(e["plate"], e["when"]),
           "car_track": lambda: x_car_track(e["plate"], e["when"]),
           "out_of_zone": lambda: x_out_of_zone(e["when"]),
           "fleet_stats": lambda: x_fleet_stats(e["when"]),
           "nearest_charger": lambda: x_nearest_charger(e["plate"], e["place"], e["when"])}[intent]
    r = run()
    if "error" not in r:                                   # จำไว้ใช้กับคำถามต่อไป
        memory["plate"] = e["plate"] or memory["plate"]
        memory["place"] = e["place"] or memory["place"]
    return intent, answer(intent, r)

DIGITS = re.compile(r"\d+")
def safe(text, facts):
    """กัน LLM แต่งตัวเลข: ทุกกลุ่มตัวเลขในคำตอบต้องมีอยู่ในข้อเท็จจริง"""
    ok = set(DIGITS.findall(facts))
    return all(d in ok for d in DIGITS.findall(text))

OPEN = ["", "ได้เลยครับ ", "เช็คให้แล้วครับ ", "ครับ "]
def friendly(facts):
    return random.choice(OPEN) + facts

def chat(q, show_facts=False):
    intent, facts = facts_for(q)
    if show_facts:
        print("  [ข้อเท็จจริง]", facts)
    if not USE_LLM:
        return friendly(facts)
    msgs = [{"role": "system", "content": SYSTEM}]
    for f_, q_, a_ in FEWSHOT:
        msgs += [{"role": "user", "content": f"ข้อมูลจากระบบ:\n{f_}\n\nคำถามของผู้ใช้: {q_}"},
                 {"role": "assistant", "content": a_}]
    msgs.append({"role": "user", "content": f"ข้อมูลจากระบบ:\n{facts}\n\nคำถามของผู้ใช้: {q}"})
    prompt = ctok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    x = ctok(prompt, return_tensors="pt").to(cmodel.device)
    with torch.no_grad():
        out = cmodel.generate(**x, max_new_tokens=200, do_sample=False, pad_token_id=ctok.eos_token_id)
    text = ctok.decode(out[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if not text or not safe(text, facts):
        if show_facts:
            print("  [ตรวจ] LLM ใส่ตัวเลขที่ไม่มีในข้อมูล → ใช้คำตอบจากระบบแทน:", text[:120])
        return friendly(facts)
    return text

def แชต():
    # พิมพ์คุยต่อเนื่อง · Enter ว่าง = จบ · พิมพ์ "ล้าง" = เริ่มบทสนทนาใหม่
    while True:
        q = input("คุณ: ").strip()
        if not q:
            break
        if q == "ล้าง":
            memory.update(plate=None, place=None); print("บอท: เริ่มใหม่แล้วครับ"); continue
        print("บอท:", chat(q))

## 11 · Chatbot ③ Claude — ใกล้ ChatGPT/Claude ที่สุด · เสียเงินตามใช้

1. สร้าง key ที่ console.anthropic.com → เติมเครดิต → **ตั้ง spend limit**
2. เมนู **Add-ons → Secrets → Add Secret** · Label `ANTHROPIC_API_KEY` · ติ๊ก **Attached** ให้ notebook นี้
3. อยากประหยัด: แก้ `MODEL = "claude-haiku-5-5"` ก่อนรัน

⚠️ ผลค้นข้อมูล (รหัสสมมติ EV-xxxx · ชื่อพื้นที่ · พิกัด) จะถูกส่งไป Anthropic · คุยต่อเนื่อง: `แชต_claude()`

In [ ]:
# ── 11 · Chatbot แบบ Claude — เข้าใจทุกรูปประโยค · ตอบสั้นตรงประเด็น · จำบทสนทนา ──
# ต้องรันส่วน 1–7 (และ 7.1 ถ้ามี) ก่อน · ไม่ต้องใช้ GPU
# ⚠️ ผลค้นข้อมูล (รหัสรถสมมติ EV-xxxx · ชื่อพื้นที่ · พิกัด) จะถูกส่งไป Anthropic — ใช้กับชุดข้อมูลทดลองนี้ได้ · ข้อมูลจริงต้องให้ DPO อนุมัติก่อน
#
# ตั้ง API key ครั้งเดียว: เมนู Add-ons → Secrets → Add Secret
#   Label = ANTHROPIC_API_KEY · Value = key จาก console.anthropic.com · ติ๊ก Attached ให้ notebook นี้
import os, sys, json, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
import anthropic

if not os.environ.get("ANTHROPIC_API_KEY"):
    from kaggle_secrets import UserSecretsClient
    os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

MODEL = "claude-opus-5-5"     # ฉลาดที่สุด · อยากเร็ว/ถูกลงลอง "claude-haiku-5-5"

# ── tools: ห่อ executor ส่วน 6 ให้ Claude เรียก · คืน JSON สั้น ๆ ─────────────
def _j(x):
    return json.dumps(x, ensure_ascii=False, default=str)

def _when(text):
    return find_time(text or "")[0]                  # "ล่าสุด" · "วันนี้" · "เมื่อวาน" · "14:30" · "9 โมง"

def _place(name):
    return find_place(name) if name else None        # เกณฑ์ 0.85 — หลวมกว่านี้ "ดาวอังคาร" จะกลายเป็น "นาวัง"

def _plate(car):
    p = find_plate(car or "")[0]
    return p or (car or "").upper().replace("-", "").replace(" ", "")

def run_tool(name, a):
    try:
        if name == "find_place":
            pl = _place(a["text"])
            return _j(pl or {"error": "ไม่พบสถานที่นี้"})
        if name == "cars_in_area":
            pl = _place(a["place"])
            if not pl:
                return _j({"error": f"ไม่รู้จักสถานที่ '{a['place']}'"})
            return _j(x_cars_in_area(pl, _when(a.get("time"))))
        if name == "car_status":
            return _j(x_car_status(_plate(a["car"]), _when(a.get("time"))))
        if name == "car_track":
            return _j(x_car_track(_plate(a["car"]), _when(a.get("time") or "วันนี้")))
        if name == "rank_areas":
            pl = _place(a.get("place"))
            return _j(x_rank_area(pl, _when(a.get("time")), top=a.get("top", 5)))
        if name == "out_of_zone":
            r = x_out_of_zone(_when(a.get("time") or "วันนี้"))
            r["cars"] = r["cars"][:15]
            return _j(r)
        if name == "fleet_stats":
            return _j(x_fleet_stats(_when(a.get("time"))))
        if name == "nearest_chargers":
            return _j(x_nearest_charger(_plate(a["car"]) if a.get("car") else None, _place(a.get("place")),
                                        _when(None), k=a.get("k", 3)))
        return _j({"error": f"ไม่มี tool ชื่อ {name}"})
    except Exception as e:                            # ส่งข้อผิดพลาดกลับให้ Claude อธิบาย/ถามใหม่ ไม่ให้ระบบล้ม
        return _j({"error": f"{type(e).__name__}: {e}"})

TIME = {"type": "string", "description": "ช่วงเวลาเป็นคำไทย เช่น 'ล่าสุด' (ค่าเริ่มต้น) · 'วันนี้' · 'เมื่อวาน' · '14:30' · '9 โมง'"}
CAR = {"type": "string", "description": "รหัสรถ เช่น EV-0042"}
PLACE = {"type": "string", "description": "ชื่อจังหวัด/อำเภอ/เขต ตามที่ผู้ใช้พิมพ์ (สะกดผิดได้)"}
def _tool(name, desc, props, required):
    return {"name": name, "description": desc,
            "input_schema": {"type": "object", "properties": props, "required": required}}

TOOLS = [
    _tool("cars_in_area", "จำนวนและรายชื่อรถที่ออนไลน์ในจังหวัด/อำเภอ/เขต ณ เวลาหนึ่ง พร้อมแยกสถานะ (จอด · ว่าง·วิ่ง · มีผู้โดยสาร)",
          {"place": PLACE, "time": TIME}, ["place"]),
    _tool("car_status", "ตำแหน่ง สถานะ ความเร็ว ทิศ ของรถ 1 คัน ณ เวลาหนึ่ง", {"car": CAR, "time": TIME}, ["car"]),
    _tool("car_track", "สรุปการเดินทางของรถ 1 คันในช่วงเวลา: ระยะทาง จุดเริ่ม-จุดสุดท้าย จังหวัดที่ผ่าน ความเร็วสูงสุด",
          {"car": CAR, "time": TIME}, ["car"]),
    _tool("rank_areas", "จัดอันดับพื้นที่ที่มีรถมากที่สุด: ใส่จังหวัด = อันดับเขต/อำเภอในจังหวัดนั้น · ไม่ใส่ = อันดับจังหวัดทั่วประเทศ",
          {"place": PLACE, "time": TIME, "top": {"type": "integer", "description": "จำนวนอันดับ (ค่าเริ่มต้น 5)"}}, []),
    _tool("out_of_zone", "รถที่ออกนอกเขตกรุงเทพฯ และปริมณฑลในช่วงเวลา พร้อมจังหวัดที่ไป", {"time": TIME}, []),
    _tool("fleet_stats", "ภาพรวมทั้งฝูงรถ: จำนวนรถออนไลน์ แยกสถานะ และ 5 จังหวัดที่รถเยอะสุด", {"time": TIME}, []),
    _tool("nearest_chargers", "สถานีชาร์จ EV ใกล้รถคันหนึ่ง หรือใกล้กลางจังหวัด (ข้อมูล OpenStreetMap มีแค่บางส่วน)",
          {"car": CAR, "place": PLACE, "k": {"type": "integer", "description": "จำนวนสถานี (ค่าเริ่มต้น 3)"}}, []),
]

SYSTEM_CLAUDE = f"""คุณคือผู้ช่วยของทีม EV7 ที่ดูแลรถแท็กซี่ไฟฟ้า คุยกับพนักงานเป็นภาษาไทย

วิธีตอบ
- ตอบเหมือนเพื่อนร่วมงานคุยกัน สั้น ตรงประเด็น ส่วนใหญ่ 1–2 ประโยค ตอบเฉพาะสิ่งที่ถาม ไม่ทวนคำถาม ไม่เกริ่น
- ถ้าเป็นรายการ แสดงไม่เกิน 5 รายการ เว้นแต่ผู้ใช้ขอเพิ่ม
- ตัวเลข ชื่อพื้นที่ และรหัสรถ ต้องมาจากผลของ tool เท่านั้น ห้ามเดา · ถ้า tool ตอบ error ให้บอกสั้น ๆ และแนะนำว่าถามยังไงแทน
- ถ้าคำถามกำกวม (เช่น ไม่ระบุว่ารถคันไหน และไม่มีในบทสนทนาก่อนหน้า) ให้ถามกลับสั้น ๆ
- ใช้บริบทจากบทสนทนาก่อนหน้า เช่น "คันนี้" "ที่นั่น"
- บอกเวลาของข้อมูลเมื่อเกี่ยวข้อง

ข้อมูลที่มี
- ตำแหน่ง GPS รถแท็กซี่ EV ช่วง 30 ก.ย. – 1 ต.ค. 2026 · "ตอนนี้" = เวลาล่าสุดของข้อมูล {LATEST:%d/%m/%Y %H:%M} (ไม่ใช่เวลาจริง)
- รหัสรถเป็นรหัสสมมติ EV-xxxx · รถออนไลน์ = มีสัญญาณภายใน 15 นาที
- "นอกเขต" = นอกกรุงเทพฯ และปริมณฑล (นนทบุรี ปทุมธานี สมุทรปราการ สมุทรสาคร นครปฐม)

ข้อห้าม
- ระบบนี้ตอบคำถามเท่านั้น สั่งการรถไม่ได้ (ตัดสัญญาณ ล็อก ดับเครื่อง) ถ้าถูกขอให้บอกว่าทำไม่ได้
- คำถามที่ไม่เกี่ยวกับรถ EV7 ให้ตอบสั้น ๆ ว่าช่วยได้เฉพาะเรื่องรถ"""

history = []          # จำบทสนทนา · history.clear() = เริ่มใหม่ · ชื่อแยกจาก cell 10 (Qwen) รันคู่กันได้

def chat_claude(q, show_tools=False):
    history.append({"role": "user", "content": q})
    for _ in range(8):                                     # กันวนไม่จบ
        resp = client.beta.messages.create(
            model=MODEL, max_tokens=2000,
            system=SYSTEM_CLAUDE, tools=TOOLS, messages=history,
            output_config={"effort": "low"},               # ตอบเร็ว · คำถามแบบนี้ไม่ต้องคิดลึก
            cache_control={"type": "ephemeral"},           # cache คำสั่ง + tools ลดค่าใช้จ่ายรอบถัดไป
            betas=["server-side-fallback-2026-06-01"],
            fallbacks=[{"model": "claude-opus-4-8"}],      # ถ้าโมเดลหลักปฏิเสธ ให้รุ่นสำรองตอบแทน
        )
        history.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason == "refusal":
            return "ขออภัย ตอบคำถามนี้ไม่ได้"
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if b.type == "text").strip()
        results = []
        for b in resp.content:
            if b.type == "tool_use":
                out = run_tool(b.name, b.input)
                if show_tools:
                    print(f"  🔧 {b.name}({json.dumps(b.input, ensure_ascii=False)}) → {out[:160]}")
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": out})
        history.append({"role": "user", "content": results})
    return "ขออภัย ใช้เวลานานเกินไป ลองถามใหม่อีกครั้ง"

def แชต_claude():
    # คุยต่อเนื่อง · Enter ว่าง = จบ · พิมพ์ "ล้าง" = เริ่มบทสนทนาใหม่
    while True:
        q = input("คุณ: ").strip()
        if not q:
            break
        if q == "ล้าง":
            history.clear(); print("บอท: เริ่มใหม่แล้วครับ"); continue
        print("บอท:", chat_claude(q))

for q in ["หวัดดี", "ตอนนี้ภูเก็ตมีรถกี่คัน", "เขตไหนในกรุงเทพมีรถเยอะสุด", "EV-2038 อยู่ไหน", "แล้ววันนี้มันไปไหนมาบ้าง", "ช่วยดับเครื่องคันนี้หน่อย"]:
    print("คุณ:", q); print("บอท:", chat_claude(q, show_tools=True)); print()

## 12 · เทียบ 3 แบบ

ถามคำถามชุดเดียวกันกับทุกแบบ พร้อมเวลาตอบ · แบบที่ยังไม่ได้รันขึ้น "ยังไม่ได้โหลด" · ถามเพิ่ม: `เทียบ("คำถาม 1", "คำถาม 2")`

In [ ]:
# ── 12 · เทียบ 3 แบบ ด้วยคำถามชุดเดียวกัน ──
# รันได้แม้มีแค่บางแบบ — แบบที่ยังไม่ได้รัน cell จะขึ้น "ยังไม่ได้โหลด"
#   ① LoRA (ฟรี)       → ask_llm()     จาก 8.9b + 8.8 + 8.10
#   ② Qwen chat (ฟรี)  → chat()        จาก cell 10
#   ③ Claude (เสียเงิน) → chat_claude() จาก cell 11
#   ④ Agent (ฟรี)      → agent()       จาก cell 13
import time

MODES = [("② Qwen", "chat"), ("③ Claude", "chat_claude"), ("④ Agent", "agent")]   # ① LoRA ส่วน 8 เลิกใช้ (adapter เก่าตอบ JSON คนละรูปแบบ)


def เทียบ(*questions):
    for q in questions:
        print("━" * 70)
        print("คุณ:", q)
        for label, fn in MODES:
            f = globals().get(fn)
            if f is None:
                print(f"  {label:9s} — ยังไม่ได้โหลด")
                continue
            t0 = time.time()
            try:
                a = f(q)
            except Exception as e:
                a = f"ERROR {type(e).__name__}: {e}"
            print(f"  {label:9s} ({time.time() - t0:4.1f} วิ) {a}")


# ล้างความจำบทสนทนาก่อนเทียบ ให้ทุกแบบเริ่มเท่ากัน
if "memory" in globals():
    memory.update(plate=None, place=None)
if "history" in globals():
    history.clear()
if "agent_history" in globals():
    agent_history.clear()

เทียบ("ตอนนี้ภูเก็ตมีรถกี่คัน",
      "เขตไหนในกรุงเทพมีรถเยอะสุด",
      "แถวโคราชมีแท็กซี่วิ่งอยู่ไหมตอนนี้",
      "รถออกนอกเขตวันนี้กี่คัน ไปไหนบ้าง",
      "ช่วยดับเครื่องรถคันนี้หน่อย",
      "พรุ่งนี้ฝนตกไหม")

## 13 · Agent ฟรี — Qwen เขียน SQL เอง (ใกล้แชตบอตจริงที่สุดแบบไม่เสียเงิน)

ต่างจาก cell 10: ไม่ต้องตกลงในช่องที่เขียนไว้ · Qwen อ่านคำอธิบายข้อมูล → เขียน SQL (DuckDB) → ดูผล → เขียนใหม่ได้หลายรอบ → สรุปเป็นภาษาคน
ถามได้ทุกเรื่องที่ข้อมูลมี เช่น "เทียบชลบุรีเมื่อวานกับวันนี้ช่วง 6 โมงเย็นถึง 3 ทุ่ม"

- ต้องรันส่วน 0–7 ก่อน · GPU T4 x2 · ครั้งแรกดาวน์โหลด Qwen 14B ~30 GB แล้วบีบ 4-bit (10–15 นาที) · ตอบข้อละ ~30–90 วินาที
- ถ้ารัน cell 10 ไว้ด้วย หน่วยความจำอาจไม่พอ → `del cmodel; torch.cuda.empty_cache()` ก่อน
- เร็วขึ้นแต่พลาดบ่อยขึ้น: `AGENT_LLM = "Qwen/Qwen2.5-7B-Instruct"`
- ดู SQL ที่ใช้: `agent("คำถาม", show_steps=True)` · คุยต่อเนื่อง: `แชต_agent()` (พิมพ์ `sql` เพื่อเปิด/ปิดการแสดง SQL)
- ตารางอ่านอย่างเดียว · รับเฉพาะ SELECT · ตัวเลขที่ไม่ได้มาจากผล query จะมี ⚠️ เตือน

In [ ]:
# ── 13 · Agent ฟรี — Qwen เขียน SQL เอง (DuckDB) · ถามอะไรก็ได้ที่ข้อมูลตอบได้ · คิดหลายขั้น ──
# ต้องรันส่วน 0–7 ก่อน (7.1 / 7.2 / 10 ไม่จำเป็น) · ใช้ GPU T4 x2 · ไม่มีค่าใช้จ่าย
#
# ต่างจาก cell 10 ยังไง
#   cell 10: คำถาม → เลือก 1 ช่องจากที่เขียนไว้ → Qwen แค่เรียบเรียง   (rule-based)
#   cell 13: คำถาม → Qwen อ่านคำอธิบายข้อมูล → เขียน SQL → ดูผล → เขียนใหม่ได้ → สรุป  (agent)
# ตัวเลขทุกตัวยังมาจากผล SQL บนข้อมูลจริง · show_steps=True แสดง SQL ที่ใช้ให้ตรวจ
import json, re, sys, time, threading, subprocess
import numpy as np
import pandas as pd
import torch

try:
    import duckdb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb

AGENT_LLM = "Qwen/Qwen2.5-14B-Instruct"   # ฟรี · 4-bit ~9 GB · เร็วกว่าแต่พลาดบ่อยกว่า: "Qwen/Qwen2.5-7B-Instruct"
MAX_STEPS = 6                             # เรียก tool ได้สูงสุดกี่รอบต่อคำถาม
MAX_ROWS = 30                             # ผล SQL ที่ส่งให้ LLM อ่าน (กันข้อความยาวเกิน)
SQL_TIMEOUT = 20                          # วินาที

# ── 1) ตารางให้ LLM query (DuckDB อ่านจาก DataFrame ในหน่วยความจำ · ไม่แก้ข้อมูลต้นทาง) ──
if "dist_code" not in chg.columns:                                   # ระบุเขตของสถานีชาร์จ (ถ้ายังไม่ได้รัน 7.2)
    from shapely.geometry import shape, Point
    from shapely import STRtree
    _polys = [shape(f["geometry"]) for f in json.load(open(OUT / "dist_pip.geojson", encoding="utf-8"))["features"]]
    _tree = STRtree(_polys)
    _idx = [(_tree.query(Point(b, a), predicate="within").tolist() or [-1])[0] for a, b in zip(chg.lat, chg.lon)]
    chg["dist_code"] = [dist.pc.iloc[i][2:] if i >= 0 else None for i in _idx]
    chg["prov_code"] = [dist.ppc.iloc[i][2:] if i >= 0 else None for i in _idx]
    chg["dist_th"] = [dist.th.iloc[i] if i >= 0 else None for i in _idx]

T_POINTS = pos[["car_id", "plate", "plate_key", "gps_time", "lat", "lon", "speed", "direction", "meter", "signal", "status",
                "dist_code", "dist_th", "prov_code", "prov_th"]].rename(columns={"gps_time": "ts"})
T_NOW = latest_at(LATEST)[["car_id", "plate", "plate_key", "gps_time", "lat", "lon", "speed", "status", "dist_code", "dist_th",
                           "prov_code", "prov_th"]].rename(columns={"gps_time": "ts"})
_s = pos.sort_values(["car_id", "gps_time"])
_f = _s.meter.eq("F")
T_PICKUPS = _s[_f & ~_f.groupby(_s.car_id).shift(fill_value=False)][["car_id", "plate", "gps_time", "lat", "lon", "dist_th", "prov_th"]] \
              .rename(columns={"gps_time": "ts"})
T_CHARGERS = chg.assign(prov_th=chg.prov_code.map(name_of))[["name", "op", "lat", "lon", "dist_code", "dist_th", "prov_code", "prov_th"]] \
               .rename(columns={"op": "operator"})
T_DISTRICTS = dist.assign(dist_code=dist.pc.str[2:], prov_code=dist.ppc.str[2:])[["dist_code", "th", "en", "prov_code", "pth"]] \
                .rename(columns={"th": "dist_th", "en": "dist_en", "pth": "prov_th"})

STATUS_SQL = {"ว่าง·วิ่ง": "วิ่งว่าง"}                     # "·" พิมพ์ยาก LLM มักเขียนผิด → ใช้ชื่อเรียบ ๆ ในตาราง SQL
for _t in (T_POINTS, T_NOW):
    _t["status"] = _t["status"].replace(STATUS_SQL)

db = duckdb.connect()
for name, df in {"points": T_POINTS, "now": T_NOW, "pickups": T_PICKUPS, "chargers": T_CHARGERS, "districts": T_DISTRICTS}.items():
    db.register(name, df)
db.execute("""CREATE MACRO dist_km(lat1, lon1, lat2, lon2) AS
              12742 * asin(sqrt(pow(sin(radians(lat2 - lat1) / 2), 2)
                     + cos(radians(lat1)) * cos(radians(lat2)) * pow(sin(radians(lon2 - lon1) / 2), 2)))""")

SCHEMA = f"""
ตาราง (DuckDB SQL)
- points  ทุกจุด GPS {len(T_POINTS):,} แถว · {pos.gps_time.min():%Y-%m-%d %H:%M} ถึง {LATEST:%Y-%m-%d %H:%M} · ~5 นาที/จุด/คัน
    car_id (รหัสประจำคัน), plate (ทะเบียน เช่น ทอ-1234), plate_key (ทะเบียนไม่มีขีด/เว้นวรรค เช่น ทอ1234), ts (timestamp), lat, lon, speed (กม./ชม.), direction (องศา 0=เหนือ 90=ตะวันออก),
    meter ('F'=มีผู้โดยสาร 'E'=ว่าง), signal ('V'=ไม่มีสัญญาณ), status = หนึ่งใน 'จอด' 'วิ่งว่าง' 'มีผู้โดยสาร' 'ไม่มีสัญญาณ' (สะกดตามนี้เท่านั้น),
    dist_code, dist_th (เขต/อำเภอ), prov_code, prov_th (จังหวัด)
- now     ตำแหน่งล่าสุดของรถที่ออนไลน์ ณ "ตอนนี้" = {LATEST:%Y-%m-%d %H:%M} ({len(T_NOW):,} คัน) · คอลัมน์เหมือน points (บางส่วน)
- pickups ทุกครั้งที่รถรับผู้โดยสาร (มิเตอร์เปลี่ยน E→F) · car_id, plate, ts, lat, lon, dist_th, prov_th
- chargers สถานีชาร์จ EV จาก OpenStreetMap {len(T_CHARGERS)} แห่ง (มีแค่บางส่วน ของจริงมากกว่านี้) · name, operator, lat, lon, dist_code, dist_th, prov_code, prov_th
- districts รายชื่อเขต/อำเภอทั้งประเทศ · dist_code, dist_th, dist_en, prov_code, prov_th
- ฟังก์ชัน dist_km(lat1, lon1, lat2, lon2) = ระยะทางเป็นกิโลเมตร

ความหมาย
- รถออนไลน์ = มีสัญญาณภายใน 15 นาที · "ตอนนี้" = {LATEST:%H:%M} ของ {LATEST:%Y-%m-%d} (ไม่ใช่เวลาจริง)
- รถว่าง = status IN ('จอด','วิ่งว่าง') (ไม่มีผู้โดยสาร) · รถวิ่ง = status IN ('วิ่งว่าง','มีผู้โดยสาร') · นอกเขต = prov_th ไม่อยู่ใน กรุงเทพมหานคร นนทบุรี ปทุมธานี สมุทรปราการ สมุทรสาคร นครปฐม
- นับจำนวนรถ ใช้ COUNT(DISTINCT car_id) เสมอ (1 คันมีหลายจุด) · ชั่วโมง: hour(ts) · วัน: CAST(ts AS DATE)
- ค้นรถตามทะเบียน: WHERE plate_key = 'ทอ1234' (ตัดขีด/เว้นวรรคจากที่ผู้ใช้พิมพ์ก่อน)
- ชื่อสถานที่ที่ผู้ใช้พิมพ์อาจสะกดผิด → เรียก find_place ก่อนเพื่อได้ชื่อ/รหัสที่ถูก แล้วกรองด้วย prov_code / dist_code
- ข้อมูลไม่มีชื่อถนน · ไม่มีข้อมูลคนขับ/รายได้/แบตเตอรี่
"""

TOOLS_A = [
    {"type": "function", "function": {
        "name": "run_sql", "description": "รัน DuckDB SQL (SELECT เท่านั้น) บนตาราง points / now / pickups / chargers / districts · คืนสูงสุด 30 แถว",
        "parameters": {"type": "object", "properties": {"sql": {"type": "string", "description": "คำสั่ง SELECT"}}, "required": ["sql"]}}},
    {"type": "function", "function": {
        "name": "find_place", "description": "แปลงชื่อจังหวัด/อำเภอ/เขตที่ผู้ใช้พิมพ์ (สะกดผิดได้) เป็นชื่อและรหัสที่ถูกต้อง",
        "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}}},
]

SYSTEM_A = f"""คุณคือผู้ช่วยวิเคราะห์ข้อมูลรถแท็กซี่ไฟฟ้า EV7 คุยกับพนักงานเป็นภาษาไทย
หาคำตอบด้วยการเรียก tool (เขียน SQL ได้หลายครั้ง ถ้าผลแปลกหรือว่างให้ตรวจ SQL แล้วลองใหม่) แล้วตอบเหมือนคนคุยกัน
- ตอบสั้น ตรงประเด็น 1–3 ประโยค ขึ้นต้นด้วยคำตอบ · พูดเวลาแบบไทยได้ (15:00 = บ่าย 3)
- ตัวเลข ชื่อ รหัสรถ ต้องมาจากผล tool เท่านั้น ห้ามเดา · ถ้าข้อมูลตอบไม่ได้ ให้บอกตรง ๆ ว่าขาดข้อมูลอะไร
- คำถามเชิงความเห็น (เช่น ควรเปิดสถานีที่ไหน) ให้หาข้อมูลประกอบก่อน แล้วบอกเหตุผลและบอกว่าเป็นการประเมินเบื้องต้น
- คำถามกำกวม ถามกลับสั้น ๆ · คำสั่งควบคุมรถ (ตัดสัญญาณ ล็อก ดับเครื่อง) ตอบประโยคเดียวว่าระบบตอบคำถามได้อย่างเดียว · เรื่องที่ไม่เกี่ยวกับรถ ตอบประโยคเดียวว่าช่วยได้เฉพาะเรื่องรถ ไม่ต้องแนะนำที่อื่น
- "วันนี้" ใช้ตาราง points กรองวัน · "ตอนนี้" ใช้ตาราง now · นับรถออกนอกเขตต่อวัน = คันที่มีจุดนอกเขตอย่างน้อย 1 จุดในวันนั้น
{SCHEMA}
ตัวอย่างการใช้ tool
ถาม: ตอนนี้เชียงใหม่มีรถกี่คัน แยกสถานะ
  1) find_place(text="เชียงใหม่") → filter: prov_code = '50'
  2) run_sql: SELECT status, COUNT(DISTINCT car_id) AS cars FROM now WHERE prov_code = '50' GROUP BY status ORDER BY cars DESC
ถาม: ชั่วโมงไหนรับผู้โดยสารมากสุด
  run_sql: SELECT hour(ts) AS h, COUNT(*) AS pickups FROM pickups GROUP BY h ORDER BY pickups DESC LIMIT 3
ถาม: ทอ-4141 วิ่งไปกี่กิโลวันที่ 1 ต.ค.
  run_sql: SELECT SUM(dist_km(lat0, lon0, lat, lon)) AS km FROM (SELECT lat, lon, lag(lat) OVER w AS lat0, lag(lon) OVER w AS lon0,
           ts - lag(ts) OVER w AS gap FROM points WHERE plate_key = 'ทอ4141' AND CAST(ts AS DATE) = '2026-10-01' AND signal <> 'V'
           WINDOW w AS (ORDER BY ts)) WHERE gap <= INTERVAL 15 MINUTE
ถาม: เทียบจำนวนรถมีผู้โดยสาร 2 วัน ช่วง 18–21 น. ในกรุงเทพ
  run_sql: SELECT CAST(ts AS DATE) AS d, COUNT(DISTINCT car_id) AS cars FROM points WHERE prov_code = '10' AND status = 'มีผู้โดยสาร'
           AND hour(ts) BETWEEN 18 AND 20 GROUP BY d ORDER BY d
แล้วตอบจากผลที่ได้ เช่น "ตอนนี้เชียงใหม่มีรถออนไลน์ 25 คัน จอด 15 วิ่งว่าง 6 มีผู้โดยสาร 4 ครับ" (ตัวเลขในตัวอย่างนี้สมมติ ห้ามใช้)"""


# ── 2) tools ─────────────────────────────────────────────────────────────
def _fmt(df):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = df[c].round(3)
        elif pd.api.types.is_datetime64_any_dtype(df[c]):
            only_date = (df[c].dt.normalize() == df[c]).all()
            df[c] = df[c].dt.strftime("%Y-%m-%d" if only_date else "%Y-%m-%d %H:%M")
    more = f"\n(แสดง {MAX_ROWS} จาก {len(df):,} แถว)" if len(df) > MAX_ROWS else ""
    return df.head(MAX_ROWS).to_csv(index=False) + more if len(df) else "(ไม่มีแถว)"


def tool_run_sql(sql):
    s = sql.strip().rstrip(";")
    if not re.match(r"(?is)^\s*(with|select)\b", s) or re.search(r"(?i)\b(insert|update|delete|drop|create|alter|attach|copy|pragma|install|load|export)\b", s):
        return "ERROR: ใช้ได้เฉพาะ SELECT"
    timer = threading.Timer(SQL_TIMEOUT, db.interrupt)
    timer.start()
    try:
        out = _fmt(db.execute(s).df())
        if out == "(ไม่มีแถว)":
            out += (" — ตรวจค่าที่กรอง: status ต้องเป็น 'จอด' 'วิ่งว่าง' 'มีผู้โดยสาร' 'ไม่มีสัญญาณ' · "
                    "สถานที่ให้ใช้ prov_code/dist_code จาก find_place · ทะเบียนใช้ plate_key ไม่มีขีด")
        return out
    except Exception as e:
        return f"ERROR: {str(e).splitlines()[0][:300]}"
    finally:
        timer.cancel()


def tool_find_place(text):
    p = find_place(text or "")
    if not p:
        return "ไม่พบสถานที่นี้"
    col = "prov_code" if p["level"] == "prov" else "dist_code"
    return json.dumps({"label": p["label"], "level": "จังหวัด" if p["level"] == "prov" else "เขต/อำเภอ",
                       "filter": f"{col} = '{p['code']}'"}, ensure_ascii=False)


def run_tool_a(name, args):
    if name == "run_sql":
        return tool_run_sql(args.get("sql", ""))
    if name == "find_place":
        return tool_find_place(args.get("text", ""))
    return f"ERROR: ไม่มี tool ชื่อ {name}"


# ── 3) โหลดโมเดล ─────────────────────────────────────────────────────────
LOAD_AGENT_MODEL = True                    # False = สร้างแค่ตาราง SQL + ฟังก์ชัน ไม่โหลดโมเดล (ใช้ก่อน cell 15–16 ที่เทรน/โหลดโมเดลเอง)
FREE_OTHERS = True                         # ปล่อยโมเดลของ cell 8 / 10 ออกจาก GPU ก่อน (กัน CUDA out of memory)
if FREE_OTHERS and LOAD_AGENT_MODEL:
    import gc
    for v in ("cmodel", "model", "base", "trainer"):
        if v in globals():
            del globals()[v]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if LOAD_AGENT_MODEL and "amodel" not in globals():
    from transformers import AutoTokenizer, AutoModelForCausalLM
    atok = AutoTokenizer.from_pretrained(AGENT_LLM)
    kw = {}
    if torch.cuda.is_available():                        # เว้นที่ ~3 GB ต่อการ์ดไว้ใช้ตอนสร้างคำตอบ
        kw["max_memory"] = {i: f"{int(torch.cuda.get_device_properties(i).total_memory / 2**30) - 3}GiB"
                            for i in range(torch.cuda.device_count())}
    if torch.cuda.is_available() and re.search(r"(7|14|32)B", AGENT_LLM):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate"], check=True)
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    amodel = AutoModelForCausalLM.from_pretrained(
        AGENT_LLM, dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None, **kw)
    amodel.eval()


def _generate(msgs, max_new=400):
    prompt = atok.apply_chat_template(msgs, tools=TOOLS_A, add_generation_prompt=True, tokenize=False)
    x = atok(prompt, return_tensors="pt").to(amodel.device)
    with torch.no_grad():
        out = amodel.generate(**x, max_new_tokens=max_new, do_sample=False, pad_token_id=atok.eos_token_id)
    return atok.decode(out[0][x["input_ids"].shape[1]:], skip_special_tokens=True).strip()


TOOL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)
agent_history = []                                         # จำบทสนทนา · agent_history.clear() = เริ่มใหม่


def agent(q, show_steps=False):
    t0 = time.time()
    agent_history.append({"role": "user", "content": q})
    msgs = [{"role": "system", "content": SYSTEM_A}] + agent_history[-12:]
    seen = []                                              # ผล tool ทั้งหมด ไว้ตรวจตัวเลข
    for step in range(MAX_STEPS):
        text = _generate(msgs)
        calls = TOOL_RE.findall(text)
        if not calls:
            break
        msgs.append({"role": "assistant", "content": text})
        for c in calls:
            try:
                j = json.loads(c)
                name, args = j.get("name"), j.get("arguments") or {}
                if isinstance(args, str):
                    args = json.loads(args)
            except Exception:
                name, args, res = "?", {}, "ERROR: tool call ไม่ใช่ JSON ที่ถูกต้อง"
            else:
                res = run_tool_a(name, args)
            seen.append(res)
            if show_steps:
                arg = args.get("sql") or args.get("text") or ""
                print(f"  🔧 {name}: {arg.strip()[:300]}\n     → {res.strip()[:300].replace(chr(10), ' | ')}")
            msgs.append({"role": "tool", "content": res})
    else:
        text = _generate(msgs + [{"role": "user", "content": "สรุปคำตอบจากข้อมูลที่มีตอนนี้ ไม่ต้องเรียก tool เพิ่ม"}])
    answer_ = re.sub(r"</?tool_(call|response)>", "", TOOL_RE.sub("", text)).strip() or "ขออภัย หาคำตอบไม่ได้ ลองถามใหม่อีกแบบ"
    # ตรวจตัวเลข: ตัวเลขในคำตอบที่ไม่พบในผล tool เลย → เตือน (LLM อาจคำนวณเอง เช่น % · หรือแต่งขึ้น)
    found = " ".join(seen) + " " + q
    nums = re.findall(r"(?<![A-Za-z\d.,\-])\d[\d,]*(?:\.\d+)?(?![A-Za-z])", answer_)
    odd = [n for n in nums if n.replace(",", "") not in found.replace(",", "")]
    if odd and seen:
        answer_ += f"\n   ⚠️ ตัวเลข {', '.join(odd[:5])} ไม่ได้มาจากผล query โดยตรง (คำนวณเองหรืออาจผิด) — ดู show_steps=True"
    agent_history.append({"role": "assistant", "content": answer_})
    if show_steps:
        print(f"  ⏱ {time.time() - t0:.0f} วิ")
    return answer_


def แชต_agent():
    # คุยต่อเนื่อง · Enter ว่าง = จบ · "ล้าง" = เริ่มใหม่ · "sql" = เปิด/ปิดการแสดง SQL
    show = False
    while True:
        q = input("คุณ: ").strip()
        if not q:
            break
        if q == "ล้าง":
            agent_history.clear(); print("บอท: เริ่มใหม่แล้วครับ"); continue
        if q.lower() == "sql":
            show = not show; print("บอท: แสดง SQL", "เปิด" if show else "ปิด"); continue
        print("บอท:", agent(q, show_steps=show))


for q in ([] if not LOAD_AGENT_MODEL else ["เวลาประมาณกี่โมงมีผู้โดยสารมากที่สุด", "มีข้อมูลของวันไหนบ้าง", "มีสถานีชาร์จกี่แห่งในกรุงเทพ",
          "เทียบชลบุรีเมื่อวานกับวันนี้ช่วง 6 โมงเย็นถึง 3 ทุ่ม รถเยอะขึ้นไหม", "ถ้าอยากเปิดปั๊ม ev ควรเปิดเขตไหน"]):
    agent_history.clear()
    print("คุณ:", q); print("บอท:", agent(q, show_steps=True)); print()

## 14 · ประเมินผลด้วยชุดคำถามจริง

1. อัปโหลด `eval_questions.csv` (จาก `scripts/taxi/qa/`) เข้า dataset `taxi-qa-data` (New Version) หรือลากใส่ `/kaggle/working`
2. แก้ `EVAL_MODES` ให้เป็นแบบที่รันไว้ (`agent` · `chat` · `chat_claude`) แล้วรัน
3. ดาวน์โหลด `eval_results_<แบบ>.csv` จากแท็บ Output → เปิดใน Excel → กรอกช่อง **ผล** (ถูก / พอใช้ / ผิด)
4. รันบางข้อ: `ประเมิน("agent", rows=[0, 9, 29])`

In [ ]:
# ── 14 · ประเมินผลด้วยชุดคำถามจริง (eval_questions.csv) ──
# อัปโหลด eval_questions.csv เข้า dataset เดียวกับข้อมูล (หรือวางใน /kaggle/working) แล้วรัน
# ผล → /kaggle/working/eval_results_<แบบ>.csv · เปิดใน Excel แล้วกรอกช่อง "ผล" (ถูก / พอใช้ / ผิด)
import time
import pandas as pd

EVAL_MODES = ["agent", "chat"]          # แบบที่จะประเมิน (ชื่อฟังก์ชัน): agent = cell 13 · chat = cell 10 · chat_claude = cell 11

src = sorted(INPUT.rglob("eval_questions.csv")) + sorted(DRIVE.glob("eval_questions.csv"))
assert src, "ไม่พบ eval_questions.csv — อัปโหลดเข้า dataset หรือ /kaggle/working ก่อน"
EVAL = pd.read_csv(src[0], encoding="utf-8-sig", dtype=str).fillna("")
print("ชุดคำถาม:", src[0], len(EVAL), "ข้อ")


def ประเมิน(mode, rows=None):
    f = globals().get(mode)
    if f is None:
        print(f"⏭ ข้าม {mode} — ยังไม่ได้รัน cell ของแบบนี้")
        return None
    out = []
    for r in (EVAL if rows is None else EVAL.iloc[rows]).to_dict("records"):   # ใช้ r["ชื่อ"] — r.ชื่อไทย พังกับสระ "ำ"
        if r["หมวด"] != "ต่อเนื่อง":                      # คำถามต่อเนื่อง ให้จำบทสนทนาจากข้อก่อนหน้า · ข้ออื่นเริ่มใหม่
            for h in ("history", "agent_history"):
                if h in globals():
                    globals()[h].clear()
            if "memory" in globals():
                memory.update(plate=None, place=None)
        t0 = time.time()
        try:
            a = f(r["คำถาม"])
        except Exception as e:
            a = f"ERROR {type(e).__name__}: {e}"
        sec = round(time.time() - t0, 1)
        out.append({"id": r["id"], "หมวด": r["หมวด"], "คำถาม": r["คำถาม"], "คำตอบ": a, "วินาที": sec,
                    "คำตอบที่อยากได้": r["คำตอบที่อยากได้"], "ควรมีในคำตอบ": r["ควรมีในคำตอบ"], "ผล": ""})
        print(f"[{r['id']}] {r['คำถาม']}\n     → {a[:200]}  ({sec} วิ)", flush=True)   # quote เดี่ยว — Python < 3.12 ห้ามซ้อน "
    res = pd.DataFrame(out)
    path = DRIVE / f"eval_results_{mode}.csv"
    res.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"\nบันทึก {path} · เวลาเฉลี่ย {res['วินาที'].mean():.1f} วิ/ข้อ")
    return res


for m in EVAL_MODES:
    ประเมิน(m)

## 15 · สร้างชุดข้อมูลสอน text-to-SQL

Claude (AI ตัวใหญ่) เขียนแบบคำถามภาษาไทย ~38 แบบ × หลายสำนวน + SQL เฉลย + รูปแบบคำตอบ → cell นี้เติมจังหวัด/เขต/ทะเบียน/เวลาจากข้อมูลจริง
แล้ว**รัน SQL ทุกข้อตรวจกับข้อมูลจริง** (ไม่ผ่าน = ทิ้ง) → ได้ ~1,300 บทสนทนา · บันทึก `taxi_sql_train.jsonl`

- แบ่ง test = 1 สำนวนต่อแบบคำถาม ที่โมเดลไม่เคยเห็นตอนเทรน → วัดว่าเข้าใจจริง ไม่ใช่จำประโยค
- ก่อนรัน: รัน cell 13 โดยตั้ง `LOAD_AGENT_MODEL = False` (ต้องการแค่ตาราง SQL ไม่ต้องโหลด 14B) · ใช้เวลา ~5 นาที
- เพิ่มแบบคำถามได้ที่ `fam(...)` · คำถามที่ไม่ต้องใช้ SQL ที่ `NO_SQL`

In [ ]:
# ── 15 · สร้างชุดข้อมูลสอน text-to-SQL (AI ตัวใหญ่เขียนโจทย์ → ตรวจกับข้อมูลจริง → ใช้สอน Qwen) ──
# ต้องรันส่วน 0–7 และ cell 13 (ตาราง SQL + db) ก่อน · ไม่ต้องใช้ GPU · ~1–3 นาที
#
# แต่ละตัวอย่าง = บทสนทนาเต็ม 1 รอบ:  ผู้ใช้ถาม → โมเดลเรียก run_sql (SQL เฉลย) → ผล SQL จริง → คำตอบภาษาไทย
# - คำถาม/SQL/รูปแบบคำตอบ เขียนโดย Claude (ครอบคลุม ~35 แบบคำถาม × หลายสำนวน) · เติมจังหวัด/เขต/ทะเบียน/เวลา จากข้อมูลจริง
# - SQL ทุกข้อถูกรันจริง · error หรือผลแปลก = ทิ้ง → เฉลยถูกเสมอ
# - แบ่ง train / test ตาม "สำนวน" (สำนวนใน test ไม่เคยเห็นตอนเทรน) → วัดว่าเข้าใจจริง ไม่ใช่จำประโยค
import json, random, re
import pandas as pd

random.seed(42)
TODAY = LATEST.normalize()
YDAY = TODAY - pd.Timedelta(days=1)
D_TODAY, D_YDAY = f"{TODAY:%Y-%m-%d}", f"{YDAY:%Y-%m-%d}"
ZONE_TH = ["กรุงเทพมหานคร", "นนทบุรี", "ปทุมธานี", "สมุทรปราการ", "สมุทรสาคร", "นครปฐม"]
ZONE_SQL = ", ".join(f"'{z}'" for z in ZONE_TH)

# ── prompt สั้นของโมเดลที่เทรน (ใช้ทั้งตอนเทรนและตอนใช้งาน · สั้นกว่า SYSTEM_A → ตอบเร็วขึ้น) ──
SYSTEM_FT = f"""คุณคือผู้ช่วยวิเคราะห์ข้อมูลรถแท็กซี่ไฟฟ้า EV7 ตอบภาษาไทยสั้น ๆ ตัวเลขต้องมาจากผล run_sql เท่านั้น
ตาราง: points(car_id, plate, plate_key, ts, lat, lon, speed, direction, meter, signal, status, dist_th, prov_th) ทุกจุด GPS
 now(คอลัมน์เดียวกัน) ตำแหน่งล่าสุดของรถออนไลน์ · pickups(car_id, plate, ts, lat, lon, dist_th, prov_th) ทุกครั้งที่รับผู้โดยสาร
 chargers(name, operator, lat, lon, dist_th, prov_th) สถานีชาร์จ OSM บางส่วน · ฟังก์ชัน dist_km(lat1, lon1, lat2, lon2)
status: 'จอด' 'วิ่งว่าง' 'มีผู้โดยสาร' 'ไม่มีสัญญาณ' · plate_key = ทะเบียนไม่มีขีด/เว้นวรรค · นับรถใช้ COUNT(DISTINCT car_id)
ตอนนี้ = {LATEST:%Y-%m-%d %H:%M} · วันนี้ = {D_TODAY} · เมื่อวาน = {D_YDAY} · นอกเขต = prov_th NOT IN ({ZONE_SQL})
ข้อมูลไม่มี: ชื่อถนน รายได้ แบตเตอรี่ ข้อมูลคนขับ · สั่งการรถไม่ได้"""
TOOLS_FT = [t for t in TOOLS_A if t["function"]["name"] == "run_sql"]

# ── ค่าที่สุ่มเติมในคำถาม (จากข้อมูลจริง) ─────────────────────────────────
_cnt = T_NOW.groupby("prov_th").car_id.nunique()
PROVS = _cnt[_cnt >= 3].index.tolist()
BKK_D = T_NOW[T_NOW.prov_th == "กรุงเทพมหานคร"].dist_th.value_counts().index[:40].tolist()
PROV_D = T_NOW[T_NOW.prov_th != "กรุงเทพมหานคร"].groupby(["prov_th", "dist_th"]).size().index.tolist()
PLATES = T_POINTS.groupby("plate").size().pipe(lambda s: s[s > 50]).index.tolist()
CHG_P = T_CHARGERS.prov_th.dropna().value_counts().index.tolist()
DAYS = [("วันนี้", D_TODAY), ("เมื่อวาน", D_YDAY), (f"วันที่ {TODAY.day} ", D_TODAY), (f"วันที่ {YDAY.day} ", D_YDAY)]   # เว้นวรรคท้าย "วันที่ 30 " ไม่ติดคำถัดไป
HOUR_TH = {6: "6 โมงเช้า", 7: "7 โมงเช้า", 8: "8 โมงเช้า", 9: "9 โมง", 10: "10 โมง", 12: "เที่ยง", 13: "บ่ายโมง", 14: "บ่าย 2",
           15: "บ่าย 3", 16: "4 โมงเย็น", 17: "5 โมงเย็น", 18: "6 โมงเย็น", 19: "1 ทุ่ม", 20: "2 ทุ่ม", 21: "3 ทุ่ม", 22: "4 ทุ่ม"}


def ptypo(name):
    # สะกดผิด/ย่อ แบบที่คนพิมพ์จริง → SQL ยังต้องใช้ชื่อที่ถูก
    alias = {"กรุงเทพมหานคร": ["กทม", "กรุงเทพ", "กรุงเทพฯ"], "นครราชสีมา": ["โคราช"], "พระนครศรีอยุธยา": ["อยุธยา"]}
    r = random.random()
    if name in alias and r < 0.5:
        return random.choice(alias[name])
    return typo(name, 0.25) if r < 0.8 else name


def plate_text(p):
    # ผู้ใช้พิมพ์ทะเบียนได้หลายแบบ: EV-0152 · EV 0152 · ev0152
    k = re.sub(r"[\s-]", "", p)
    return random.choice([p, p, p.replace("-", " "), k.lower() if k.isascii() else k])


def pkey(p):
    return re.sub(r"[\s-]", "", p).upper()


def th_h(h):
    return HOUR_TH.get(h, f"{h}:00")


def n(x):
    return f"{int(x):,}"


def rows_txt(df, unit="คัน", k=5):
    return " · ".join(f"{i}. {r[0]} {n(r[1])} {unit}" for i, r in enumerate(df.head(k).itertuples(index=False), 1))


# ── แบบคำถาม: (ชื่อ, [สำนวน], สร้างพารามิเตอร์, SQL, สร้างคำตอบจากผล) ─────────────
F = []
def fam(name, qs, gen, sql, ans):
    F.append(dict(name=name, qs=qs, gen=gen, sql=sql, ans=ans))

fam("cars_now_prov",
    ["ตอนนี้{p}มีรถกี่คัน", "{p}มีแท็กซี่ออนไลน์กี่คัน", "รถเราอยู่{p}ตอนนี้กี่คัน", "ขอจำนวนรถใน{p}หน่อย", "{p} มีรถอยู่เท่าไหร่", "แถว{p}มีรถไหม"],
    lambda: {"P": random.choice(PROVS)},
    "SELECT COUNT(DISTINCT car_id) AS cars FROM now WHERE prov_th = '{P}'",
    lambda d, a: f"ตอนนี้{a['P']}มีรถออนไลน์ {n(d.iloc[0, 0])} คันครับ")
fam("status_now_prov",
    ["{p}ตอนนี้มีรถว่างกี่คัน มีผู้โดยสารกี่คัน", "แยกสถานะรถใน{p}ให้หน่อย", "รถใน{p}จอดกี่คัน วิ่งกี่คัน", "{p}มีรถกี่คัน ว่างกี่คัน มีคนนั่งกี่คัน", "สถานะรถ{p}ตอนนี้"],
    lambda: {"P": random.choice(PROVS)},
    "SELECT status, COUNT(DISTINCT car_id) AS cars FROM now WHERE prov_th = '{P}' GROUP BY status ORDER BY cars DESC",
    lambda d, a: f"{a['P']}มีรถ {n(d.cars.sum())} คัน: " + " · ".join(f"{s} {n(c)}" for s, c in zip(d.status, d.cars)) + " ครับ")
fam("cars_now_bkk_dist",
    ["เขต{d}มีรถกี่คัน", "ตอนนี้แถว{d}มีแท็กซี่กี่คัน", "{d}มีรถว่างไหม", "รถในเขต{d}ตอนนี้"],
    lambda: {"D": random.choice(BKK_D)},
    "SELECT status, COUNT(DISTINCT car_id) AS cars FROM now WHERE prov_th = 'กรุงเทพมหานคร' AND dist_th = '{D}' GROUP BY status ORDER BY cars DESC",
    lambda d, a: f"เขต{a['D']}มีรถ {n(d.cars.sum())} คัน (" + " · ".join(f"{s} {n(c)}" for s, c in zip(d.status, d.cars)) + ")")
fam("moving_now_prov",
    ["แถว{p}มีแท็กซี่วิ่งอยู่ไหมตอนนี้", "ตอนนี้{p}มีรถวิ่งอยู่กี่คัน", "{p}มีรถกำลังวิ่งไหม", "รถที่วิ่งอยู่ใน{p}ตอนนี้"],
    lambda: {"P": random.choice(PROVS)},
    "SELECT COUNT(DISTINCT car_id) FILTER (WHERE status IN ('วิ่งว่าง', 'มีผู้โดยสาร')) AS moving, COUNT(DISTINCT car_id) AS online FROM now WHERE prov_th = '{P}'",
    lambda d, a: f"{a['P']}ตอนนี้มีรถกำลังวิ่ง {n(d.moving[0])} คัน จากรถออนไลน์ {n(d.online[0])} คันครับ")
fam("out_of_zone_day_where",
    ["{day}รถออกนอกเขตกี่คัน ไปไหนบ้าง", "รถออกนอกเขต{day}ไปจังหวัดไหนบ้าง กี่คัน", "{day}มีรถหลุดโซนกี่คัน อยู่จังหวัดไหน"],
    lambda: dict(zip(("DAYW", "DAY"), random.choice(DAYS))),
    "SELECT prov_th, COUNT(DISTINCT car_id) AS cars, (SELECT COUNT(DISTINCT car_id) FROM points WHERE CAST(ts AS DATE) = '{DAY}' AND signal <> 'V' AND prov_th NOT IN (" + ZONE_SQL + ")) AS total FROM points WHERE CAST(ts AS DATE) = '{DAY}' AND signal <> 'V' AND prov_th NOT IN (" + ZONE_SQL + ") GROUP BY prov_th ORDER BY cars DESC LIMIT 5",
    lambda d, a: f"{a['DAYW']}มีรถออกนอกกรุงเทพฯ และปริมณฑล {n(d.total[0])} คัน ไปมากสุดที่ " + rows_txt(d[["prov_th", "cars"]], k=3))
fam("hired_now_prov",
    ["{p}มีรถรับผู้โดยสารอยู่กี่คัน", "ตอนนี้{p}มีรถที่มีผู้โดยสารกี่คัน", "รถไม่ว่างใน{p}กี่คัน"],
    lambda: {"P": random.choice(PROVS)},
    "SELECT COUNT(DISTINCT car_id) AS cars FROM now WHERE prov_th = '{P}' AND status = 'มีผู้โดยสาร'",
    lambda d, a: f"{a['P']}มีรถที่มีผู้โดยสาร {n(d.iloc[0, 0])} คันครับ")
fam("idle_now_prov",
    ["รถว่างใน{p}มีกี่คัน", "{p}มีรถว่างพร้อมรับงานกี่คัน", "ตอนนี้{p}รถว่างเท่าไหร่"],
    lambda: {"P": random.choice(PROVS)},
    "SELECT COUNT(DISTINCT car_id) AS cars FROM now WHERE prov_th = '{P}' AND status IN ('จอด', 'วิ่งว่าง')",
    lambda d, a: f"{a['P']}มีรถว่าง (จอดหรือวิ่งว่าง) {n(d.iloc[0, 0])} คันครับ")
fam("fleet_now",
    ["ตอนนี้รถออนไลน์ทั้งหมดกี่คัน", "ภาพรวมรถตอนนี้", "ทั้งระบบมีรถกี่คัน แยกสถานะ", "สรุปสถานะรถทั้งหมด", "รถทั้งฝูงตอนนี้เป็นยังไง"],
    lambda: {},
    "SELECT status, COUNT(DISTINCT car_id) AS cars FROM now GROUP BY status ORDER BY cars DESC",
    lambda d, a: f"ตอนนี้รถออนไลน์ {n(d.cars.sum())} คัน: " + " · ".join(f"{s} {n(c)}" for s, c in zip(d.status, d.cars)))
fam("top_prov_now",
    ["จังหวัดไหนมีรถมากที่สุด", "5 จังหวัดที่รถเยอะสุด", "จัดอันดับจังหวัดตามจำนวนรถ", "รถกระจุกอยู่จังหวัดไหน"],
    lambda: {},
    "SELECT prov_th, COUNT(DISTINCT car_id) AS cars FROM now GROUP BY prov_th ORDER BY cars DESC LIMIT 5",
    lambda d, a: "จังหวัดที่มีรถมากที่สุด: " + rows_txt(d))
fam("top_dist_in_prov",
    ["เขตไหนใน{p}มีรถมากที่สุด", "อำเภอไหนใน{p}รถเยอะสุด", "จัดอันดับพื้นที่ใน{p}ตามจำนวนรถ", "ใน{p} รถอยู่แถวไหนเยอะ"],
    lambda: {"P": random.choice(["กรุงเทพมหานคร"] * 3 + PROVS[:15])},
    "SELECT dist_th, COUNT(DISTINCT car_id) AS cars FROM now WHERE prov_th = '{P}' GROUP BY dist_th ORDER BY cars DESC LIMIT 5",
    lambda d, a: f"พื้นที่ใน{a['P']}ที่มีรถมากที่สุด: " + rows_txt(d))
fam("car_where",
    ["{pl} อยู่ไหน", "รถ {pl} ตอนนี้อยู่ที่ไหน", "ตำแหน่งล่าสุดของ {pl}", "{pl} วิ่งอยู่หรือจอด", "เช็คสถานะ {pl} หน่อย"],
    lambda: {"PL": random.choice(PLATES)},
    "SELECT plate, ts, status, speed, dist_th, prov_th FROM points WHERE plate_key = '{K}' ORDER BY ts DESC LIMIT 1",
    lambda d, a: f"{d.plate[0]} อยู่ที่{d.dist_th[0]} {d.prov_th[0]} สถานะ{d.status[0]} (จุดล่าสุด {pd.Timestamp(d.ts[0]):%d/%m %H:%M})")
fam("car_km_day",
    ["{pl} {day}วิ่งไปกี่กิโล", "{day} {pl} วิ่งไปกี่กิโลเมตร", "ระยะทางของ {pl} {day}", "{pl} วิ่งได้กี่โล{day}"],
    lambda: dict(zip(("DAYW", "DAY"), random.choice(DAYS)), PL=random.choice(PLATES)),
    """SELECT ROUND(SUM(dist_km(lat0, lon0, lat, lon)), 1) AS km FROM (SELECT lat, lon, lag(lat) OVER w AS lat0, lag(lon) OVER w AS lon0,
 ts - lag(ts) OVER w AS gap FROM points WHERE plate_key = '{K}' AND CAST(ts AS DATE) = '{DAY}' AND signal <> 'V' WINDOW w AS (ORDER BY ts))
 WHERE gap <= INTERVAL 15 MINUTE""",
    lambda d, a: f"{a['PL']} {a['DAYW']}วิ่งประมาณ {d.km[0]} กม. ครับ (คิดจากระยะเส้นตรงระหว่างจุด GPS)")
fam("car_provinces",
    ["{pl} ไปจังหวัดไหนมาบ้าง", "{pl} ผ่านจังหวัดอะไรบ้าง", "{pl} ออกต่างจังหวัดไหม"],
    lambda: {"PL": random.choice(PLATES)},
    "SELECT prov_th, COUNT(*) AS points, MIN(ts) AS first_seen FROM points WHERE plate_key = '{K}' AND signal <> 'V' GROUP BY prov_th ORDER BY first_seen",
    lambda d, a: f"{a['PL']} อยู่ใน " + ", ".join(d.prov_th) + ("" if len(d) > 1 else " จังหวัดเดียว ไม่ได้ออกต่างจังหวัด"))
fam("car_pickups",
    ["{pl} รับผู้โดยสารกี่ครั้ง", "{pl} ได้งานกี่เที่ยว", "{pl} รับลูกค้าไปกี่รอบ"],
    lambda: {"PL": random.choice(PLATES)},
    "SELECT COUNT(*) AS pickups FROM pickups WHERE plate = '{PL}'",
    lambda d, a: f"{a['PL']} รับผู้โดยสาร {n(d.pickups[0])} ครั้งตลอดช่วงข้อมูลครับ")
fam("peak_hour_pickups",
    ["ชั่วโมงไหนรับผู้โดยสารมากที่สุด", "ช่วงไหนคนเรียกรถเยอะสุด", "เวลาไหนมีงานเยอะ", "peak hour ของการรับผู้โดยสาร"],
    lambda: {},
    "SELECT hour(ts) AS h, COUNT(*) AS pickups FROM pickups GROUP BY h ORDER BY pickups DESC LIMIT 3",
    lambda d, a: f"รับผู้โดยสารมากสุดช่วง {th_h(int(d.h[0]))} ({n(d.pickups[0])} ครั้ง) รองลงมา {th_h(int(d.h[1]))} และ {th_h(int(d.h[2]))}")
fam("peak_hour_hired",
    ["เวลากี่โมงมีผู้โดยสารมากที่สุด", "ช่วงไหนรถมีผู้โดยสารเยอะสุด", "ชั่วโมงไหนรถไม่ว่างมากสุด"],
    lambda: {},
    "SELECT hour(ts) AS h, COUNT(DISTINCT car_id) AS cars FROM points WHERE status = 'มีผู้โดยสาร' GROUP BY h ORDER BY cars DESC LIMIT 3",
    lambda d, a: f"ช่วง {th_h(int(d.h[0]))} มีรถที่มีผู้โดยสารมากที่สุด {n(d.cars[0])} คัน")
fam("pickups_prov_day",
    ["{day}{p}รับผู้โดยสารไปกี่ครั้ง", "{p}{day}มีงานกี่เที่ยว", "จำนวนการรับผู้โดยสารใน{p}{day}"],
    lambda: dict(zip(("DAYW", "DAY"), random.choice(DAYS)), P=random.choice(PROVS[:12])),
    "SELECT COUNT(*) AS pickups FROM pickups WHERE prov_th = '{P}' AND CAST(ts AS DATE) = '{DAY}'",
    lambda d, a: f"{a['DAYW']}{a['P']}มีการรับผู้โดยสาร {n(d.pickups[0])} ครั้ง")
fam("top_pickup_dist",
    ["เขตไหนใน{p}คนเรียกรถมากสุด", "จุดรับผู้โดยสารยอดนิยมใน{p}อยู่เขตไหน", "พื้นที่ไหนใน{p}ได้งานเยอะ"],
    lambda: {"P": random.choice(["กรุงเทพมหานคร"] * 3 + PROVS[:8])},
    "SELECT dist_th, COUNT(*) AS pickups FROM pickups WHERE prov_th = '{P}' GROUP BY dist_th ORDER BY pickups DESC LIMIT 5",
    lambda d, a: f"พื้นที่ใน{a['P']}ที่รับผู้โดยสารมากสุด: " + rows_txt(d, "ครั้ง"))
fam("top_cars_pickups",
    ["รถคันไหนรับผู้โดยสารบ่อยสุด", "คันไหนได้งานเยอะที่สุด", "5 อันดับรถที่รับลูกค้ามากสุด", "คนขับคันไหนขยันสุด"],
    lambda: {},
    "SELECT plate, COUNT(*) AS pickups FROM pickups GROUP BY plate ORDER BY pickups DESC LIMIT 5",
    lambda d, a: "รถที่รับผู้โดยสารบ่อยสุด: " + rows_txt(d, "ครั้ง"))
fam("data_range",
    ["มีข้อมูลของวันไหนบ้าง", "ข้อมูลเริ่มวันไหนถึงวันไหน", "ข้อมูลล่าสุดถึงเมื่อไหร่", "ข้อมูลมีกี่วัน"],
    lambda: {},
    "SELECT MIN(ts) AS first_ts, MAX(ts) AS last_ts, COUNT(DISTINCT car_id) AS cars FROM points",
    lambda d, a: f"มีข้อมูลตั้งแต่ {pd.Timestamp(d.first_ts[0]):%d/%m/%Y %H:%M} ถึง {pd.Timestamp(d.last_ts[0]):%d/%m/%Y %H:%M} รวม {n(d.cars[0])} คัน")
fam("online_at_hour",
    ["ตอน{h} {day}มีรถออนไลน์กี่คัน", "{day}ช่วง{h}มีรถวิ่งกี่คัน", "{day} {h} มีรถในระบบเท่าไหร่"],
    lambda: dict(zip(("DAYW", "DAY"), random.choice(DAYS)), H=random.choice(list(HOUR_TH))),
    "SELECT COUNT(DISTINCT car_id) AS cars FROM points WHERE CAST(ts AS DATE) = '{DAY}' AND hour(ts) = {H} AND signal <> 'V'",
    lambda d, a: f"{a['DAYW']}ช่วง{th_h(a['H'])} มีรถส่งสัญญาณ {n(d.cars[0])} คัน")
fam("compare_days_prov",
    ["เทียบ{p}เมื่อวานกับวันนี้ช่วง{h}ถึง{h2} รถเยอะขึ้นไหม", "{p} ช่วง{h}-{h2} เมื่อวานกับวันนี้ต่างกันยังไง", "เปรียบเทียบจำนวนรถใน{p} 2 วัน ช่วง{h}ถึง{h2}"],
    lambda: (lambda h: {"P": random.choice(PROVS[:12]), "H": h, "H2": h + random.choice([2, 3])})(random.choice([7, 8, 12, 13, 17, 18])),
    "SELECT CAST(ts AS DATE) AS d, COUNT(DISTINCT car_id) AS cars FROM points WHERE prov_th = '{P}' AND hour(ts) BETWEEN {H} AND {H2m} AND signal <> 'V' GROUP BY d ORDER BY d",
    lambda d, a: f"{a['P']} ช่วง{th_h(a['H'])}–{th_h(a['H2'])}: " + " · ".join(f"{pd.Timestamp(x):%d/%m} {n(c)} คัน" for x, c in zip(d.d, d.cars)))
fam("chargers_prov",
    ["{p}มีสถานีชาร์จกี่แห่ง", "ที่ชาร์จ EV ใน{p}มีกี่ที่", "จำนวนปั๊มอีวีใน{p}", "{p}มีจุดชาร์จรถไฟฟ้าไหม"],
    lambda: {"P": random.choice(CHG_P[:25] + PROVS[:10])},
    "SELECT COUNT(*) AS chargers FROM chargers WHERE prov_th = '{P}'",
    lambda d, a: f"{a['P']}มีสถานีชาร์จ {n(d.chargers[0])} แห่ง (ข้อมูล OpenStreetMap มีแค่บางส่วน)")
fam("chargers_operator",
    ["ผู้ให้บริการสถานีชาร์จเจ้าไหนมีมากสุด", "แบรนด์ที่ชาร์จไหนเยอะที่สุด", "สถานีชาร์จแยกตามผู้ให้บริการ"],
    lambda: {},
    "SELECT operator, COUNT(*) AS chargers FROM chargers WHERE operator <> '' GROUP BY operator ORDER BY chargers DESC LIMIT 5",
    lambda d, a: "ผู้ให้บริการที่มีสถานีมากสุด: " + rows_txt(d, "แห่ง"))
fam("nearest_charger_car",
    ["{pl} ใกล้สถานีชาร์จไหนที่สุด", "ที่ชาร์จใกล้ {pl} อยู่ไหน", "{pl} ควรไปชาร์จที่ไหน"],
    lambda: {"PL": random.choice(PLATES)},
    """SELECT c.name, c.operator, c.dist_th, ROUND(dist_km(p.lat, p.lon, c.lat, c.lon), 1) AS km
 FROM (SELECT lat, lon FROM points WHERE plate_key = '{K}' AND signal <> 'V' ORDER BY ts DESC LIMIT 1) p, chargers c ORDER BY km LIMIT 3""",
    lambda d, a: f"สถานีชาร์จใกล้ {a['PL']} ที่สุด: {d.name[0]} ({d.dist_th[0]}) {d.km[0]} กม.")
fam("cars_near_chargers",
    ["มีรถจอดอยู่ที่สถานีชาร์จกี่คัน", "รถจอดใกล้ที่ชาร์จกี่คัน", "ตอนนี้รถอยู่ที่จุดชาร์จกี่คัน"],
    lambda: {},
    """SELECT COUNT(DISTINCT n.car_id) AS cars FROM now n JOIN chargers c ON dist_km(n.lat, n.lon, c.lat, c.lon) <= 0.3 WHERE n.status = 'จอด'""",
    lambda d, a: f"มีรถจอดห่างสถานีชาร์จไม่เกิน 300 ม. ประมาณ {n(d.cars[0])} คัน (ประมาณจากระยะ ไม่รู้ว่ากำลังชาร์จจริงไหม)")
fam("out_of_zone_day",
    ["{day}มีรถออกนอกเขตกี่คัน", "{day}รถวิ่งออกต่างจังหวัดกี่คัน", "นับรถที่ออกนอกกรุงเทพและปริมณฑล{day}"],
    lambda: dict(zip(("DAYW", "DAY"), random.choice(DAYS))),
    "SELECT COUNT(DISTINCT car_id) AS cars FROM points WHERE CAST(ts AS DATE) = '{DAY}' AND signal <> 'V' AND prov_th NOT IN (" + ZONE_SQL + ")",
    lambda d, a: f"{a['DAYW']}มีรถอยู่นอกกรุงเทพฯ และปริมณฑล {n(d.cars[0])} คัน")
fam("out_of_zone_where",
    ["รถที่ออกนอกเขตไปจังหวัดไหนบ้าง", "รถนอกเขตอยู่จังหวัดไหนมากสุด", "จังหวัดนอกเขตที่มีรถเราไป"],
    lambda: {},
    "SELECT prov_th, COUNT(DISTINCT car_id) AS cars FROM points WHERE signal <> 'V' AND prov_th NOT IN (" + ZONE_SQL + ") GROUP BY prov_th ORDER BY cars DESC LIMIT 5",
    lambda d, a: "จังหวัดนอกเขตที่มีรถไปมากสุด: " + rows_txt(d))
fam("no_signal_now",
    ["ตอนนี้รถไม่มีสัญญาณกี่คัน", "รถที่สัญญาณหายอยู่แถวไหน", "มีรถขาดการติดต่อไหม"],
    lambda: {},
    "SELECT prov_th, COUNT(DISTINCT car_id) AS cars FROM now WHERE status = 'ไม่มีสัญญาณ' GROUP BY prov_th ORDER BY cars DESC LIMIT 5",
    lambda d, a: f"รถไม่มีสัญญาณ {n(d.cars.sum())} คัน มากสุดที่ " + rows_txt(d, k=3))
fam("avg_speed_prov",
    ["รถใน{p}วิ่งเร็วเฉลี่ยเท่าไหร่", "ความเร็วเฉลี่ยตอนวิ่งใน{p}", "{p}รถวิ่งกี่กิโลต่อชั่วโมงโดยเฉลี่ย"],
    lambda: {"P": random.choice(PROVS[:15])},
    "SELECT ROUND(AVG(speed), 1) AS avg_kmh FROM points WHERE prov_th = '{P}' AND speed >= 5 AND signal <> 'V'",
    lambda d, a: f"ตอนวิ่ง รถใน{a['P']}เร็วเฉลี่ย {d.avg_kmh[0]} กม./ชม.")
fam("status_ratio",
    ["เทียบสัดส่วนรถที่วิ่งกับรถที่จอด", "รถจอดกี่เปอร์เซ็นต์", "สัดส่วนรถมีผู้โดยสารต่อรถว่าง", "อัตราส่วนรถวิ่งต่อรถจอด"],
    lambda: {},
    """SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE status IN ('วิ่งว่าง', 'มีผู้โดยสาร')) / COUNT(*)) AS moving_pct,
 ROUND(100.0 * COUNT(*) FILTER (WHERE status = 'จอด') / COUNT(*)) AS parked_pct,
 ROUND(100.0 * COUNT(*) FILTER (WHERE status = 'มีผู้โดยสาร') / COUNT(*)) AS hired_pct FROM now""",
    lambda d, a: f"ตอนนี้รถวิ่ง {int(d.moving_pct[0])}% จอด {int(d.parked_pct[0])}% (มีผู้โดยสาร {int(d.hired_pct[0])}%)")
fam("prov_no_charger",
    ["จังหวัดไหนมีรถแต่ไม่มีสถานีชาร์จ", "จังหวัดที่ไม่มีที่ชาร์จเลยแต่มีรถวิ่ง", "ที่ไหนขาดสถานีชาร์จ"],
    lambda: {},
    """SELECT n.prov_th, COUNT(DISTINCT n.car_id) AS cars FROM now n WHERE n.prov_th NOT IN (SELECT DISTINCT prov_th FROM chargers WHERE prov_th IS NOT NULL)
 GROUP BY n.prov_th ORDER BY cars DESC LIMIT 5""",
    lambda d, a: "จังหวัดที่มีรถแต่ไม่พบสถานีชาร์จ (OSM): " + rows_txt(d))
fam("site_recommend",
    ["ถ้าอยากเปิดปั๊ม EV ใน{p} ควรเปิดเขตไหน", "ควรตั้งสถานีชาร์จที่ไหนดีใน{p}", "ทำเลไหนใน{p}เหมาะเปิดจุดชาร์จ"],
    lambda: {"P": random.choice(["กรุงเทพมหานคร"] * 3 + PROVS[:6])},
    """SELECT a.dist_th, a.cars_per_hour, COALESCE(c.chargers, 0) AS chargers, ROUND(a.cars_per_hour / (COALESCE(c.chargers, 0) + 1), 1) AS score
 FROM (SELECT dist_th, ROUND(COUNT(DISTINCT car_id || CAST(date_trunc('hour', ts) AS VARCHAR)) / COUNT(DISTINCT date_trunc('hour', ts)), 1) AS cars_per_hour
       FROM points WHERE prov_th = '{P}' AND signal <> 'V' GROUP BY dist_th) a
 LEFT JOIN (SELECT dist_th, COUNT(*) AS chargers FROM chargers WHERE prov_th = '{P}' GROUP BY dist_th) c USING (dist_th)
 ORDER BY score DESC LIMIT 3""",
    lambda d, a: f"แนะนำ{d.dist_th[0]} (รถเฉลี่ย {d.cars_per_hour[0]} คัน/ชม. มีสถานี {int(d.chargers[0])} แห่ง) รองลงมา{d.dist_th[1]} — ประเมินเบื้องต้นจากจำนวนรถกับสถานี OSM")
fam("longest_km_day",
    ["รถคันไหนวิ่งไกลสุด{day}", "{day}คันไหนวิ่งเยอะที่สุด", "5 อันดับรถที่วิ่งไกลสุด{day}"],
    lambda: dict(zip(("DAYW", "DAY"), random.choice(DAYS))),
    """SELECT plate, ROUND(SUM(dist_km(lat0, lon0, lat, lon)), 1) AS km FROM (SELECT plate, lat, lon, lag(lat) OVER w AS lat0, lag(lon) OVER w AS lon0,
 ts - lag(ts) OVER w AS gap FROM points WHERE CAST(ts AS DATE) = '{DAY}' AND signal <> 'V' WINDOW w AS (PARTITION BY car_id ORDER BY ts))
 WHERE gap <= INTERVAL 15 MINUTE GROUP BY plate ORDER BY km DESC LIMIT 5""",
    lambda d, a: f"{a['DAYW']}รถที่วิ่งไกลสุด: " + rows_txt(d, "กม."))

# ── เพิ่มจากคลังคำถามธุรกิจ (EV7 Taxi - Business Questions · สถานะ 🟡) ──
fam("offline_long",
    ["รถคันไหนไม่ส่งสัญญาณเกิน {hr} ชั่วโมง", "มีรถหายจากระบบนานกว่า {hr} ชม. กี่คัน", "รถที่ขาดการติดต่อเกิน {hr} ชั่วโมง"],
    lambda: {"HR": random.choice([1, 2, 3, 6, 12])},
    f"""SELECT COUNT(*) AS cars, MIN(last_ts) AS oldest FROM (SELECT car_id, MAX(ts) AS last_ts FROM points WHERE signal <> 'V' GROUP BY car_id)
 WHERE last_ts < TIMESTAMP '{LATEST:%Y-%m-%d %H:%M:%S}' - INTERVAL {{HR}} HOUR""",
    lambda d, a: f"มีรถไม่ส่งสัญญาณเกิน {a['HR']} ชั่วโมง {n(d.cars[0])} คัน")
fam("speeding",
    ["รถคันไหนวิ่งเร็วเกิน {spd} กม./ชม.", "มีรถขับเร็วเกิน {spd} กี่คัน", "{day}รถคันไหนขับเร็วสุด"],
    lambda: dict(zip(("DAYW", "DAY"), random.choice(DAYS)), SPD=random.choice([100, 110, 120])),
    "SELECT plate, MAX(speed) AS max_kmh FROM points WHERE CAST(ts AS DATE) = '{DAY}' AND signal <> 'V' AND speed > {SPD} GROUP BY plate ORDER BY max_kmh DESC LIMIT 5",
    lambda d, a: f"{a['DAYW']}มีรถวิ่งเร็วเกิน {a['SPD']} กม./ชม. — เร็วสุด: " + rows_txt(d, "กม./ชม."))
fam("car_out_of_zone",
    ["{pl} ออกนอกเขตไหม", "{pl} ไปนอกกรุงเทพและปริมณฑลหรือเปล่า", "{pl} เคยออกต่างจังหวัดนอกโซนไหม"],
    lambda: {"PL": random.choice(PLATES)},
    "SELECT COUNT(*) AS points_outside, COUNT(DISTINCT prov_th) AS provinces, string_agg(DISTINCT prov_th, ', ') AS where_ FROM points WHERE plate_key = '{K}' AND signal <> 'V' AND prov_th NOT IN (" + ZONE_SQL + ")",
    lambda d, a: (f"{a['PL']} ออกนอกเขต ไปที่ {d.where_[0]}" if d.points_outside[0] else f"{a['PL']} ไม่เคยออกนอกกรุงเทพฯ และปริมณฑลในช่วงข้อมูล"))
fam("fleet_km_day",
    ["{day}รถทั้งหมดวิ่งรวมกี่กิโล", "ระยะทางรวมทั้งฝูง{day}", "{day}เฉลี่ยรถวิ่งคันละกี่กิโล"],
    lambda: dict(zip(("DAYW", "DAY"), random.choice(DAYS))),
    """SELECT ROUND(SUM(km)) AS total_km, ROUND(AVG(km), 1) AS avg_km_per_car, COUNT(*) AS cars FROM (
 SELECT car_id, SUM(dist_km(lat0, lon0, lat, lon)) AS km FROM (SELECT car_id, lat, lon, lag(lat) OVER w AS lat0, lag(lon) OVER w AS lon0,
 ts - lag(ts) OVER w AS gap FROM points WHERE CAST(ts AS DATE) = '{DAY}' AND signal <> 'V' WINDOW w AS (PARTITION BY car_id ORDER BY ts))
 WHERE gap <= INTERVAL 15 MINUTE GROUP BY car_id)""",
    lambda d, a: f"{a['DAYW']}รถ {n(d.cars[0])} คันวิ่งรวม {n(d.total_km[0])} กม. เฉลี่ยคันละ {d.avg_km_per_car[0]} กม.")
fam("utilization_prov",
    ["อัตราการใช้งานรถใน{p}เท่าไหร่", "รถใน{p}มีผู้โดยสารกี่เปอร์เซ็นต์ของเวลา", "{p}รถทำงานคุ้มไหม มีผู้โดยสารบ่อยแค่ไหน"],
    lambda: {"P": random.choice(PROVS[:15])},
    "SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE status = 'มีผู้โดยสาร') / COUNT(*), 1) AS hired_pct, COUNT(DISTINCT car_id) AS cars FROM points WHERE prov_th = '{P}' AND signal <> 'V'",
    lambda d, a: f"รถใน{a['P']}มีผู้โดยสาร {d.hired_pct[0]}% ของจุดข้อมูลทั้งหมด (รถ {n(d.cars[0])} คัน)")
fam("gps_gap_moving",
    ["รถคันไหนสัญญาณหายระหว่างวิ่ง", "สงสัยถอด GPS คันไหนบ้าง", "รถที่สัญญาณขาดตอนกำลังวิ่งเร็ว"],
    lambda: {},
    """SELECT plate, COUNT(*) AS gaps, MAX(gap_min) AS max_gap_min FROM (SELECT plate, speed,
 date_diff('minute', ts, lead(ts) OVER (PARTITION BY car_id ORDER BY ts)) AS gap_min FROM points WHERE signal <> 'V')
 WHERE speed >= 40 AND gap_min >= 30 GROUP BY plate ORDER BY gaps DESC, max_gap_min DESC LIMIT 5""",
    lambda d, a: "รถที่สัญญาณหายเกิน 30 นาทีขณะวิ่ง ≥ 40 กม./ชม. (อาจอับสัญญาณหรือถอด GPS — ต้องตรวจต่อ): " + rows_txt(d, "ครั้ง"))

# คำถามที่ไม่ต้องเรียก SQL (ปฏิเสธ / ข้อมูลไม่มี / นอกเรื่อง)
NO_SQL = [
    (["ช่วยดับเครื่อง {pl} หน่อย", "ตัดสัญญาณรถ {pl}", "ล็อครถ {pl} เลย", "สั่งหยุดรถ {pl}"],
     "ระบบนี้ตอบคำถามจากข้อมูลได้อย่างเดียว สั่งการรถ (ดับเครื่อง ตัดสัญญาณ ล็อก) ไม่ได้ครับ"),
    (["คนขับคันไหนได้เงินเยอะสุด", "รายได้ต่อคันเท่าไหร่", "วันนี้รถทำเงินได้เท่าไหร่"],
     "ข้อมูลที่มีเป็นแค่ตำแหน่ง GPS กับสถานะมิเตอร์ ไม่มีข้อมูลรายได้ครับ · ถ้าต้องการ ดูจำนวนครั้งที่รับผู้โดยสารแทนได้"),
    (["รถคันไหนแบตใกล้หมด", "แบตเหลือกี่เปอร์เซ็นต์", "คันไหนต้องชาร์จด่วน"],
     "ข้อมูลไม่มีระดับแบตเตอรี่ครับ (ต้องขอจากระบบ SMART)"),
    (["ถนนสุขุมวิทตอนนี้มีรถกี่คัน", "รถบนถนนพระราม 9 มีกี่คัน", "ถนนไหนรถติด"],
     "ข้อมูลไม่มีชื่อถนนครับ ถามเป็นเขตหรืออำเภอแทนได้ เช่น 'เขตวัฒนามีรถกี่คัน'"),
    (["พรุ่งนี้ฝนตกไหม", "ราคาหุ้นวันนี้", "ช่วยเขียนอีเมลให้หน่อย", "กินอะไรดี"],
     "ช่วยได้เฉพาะเรื่องข้อมูลรถแท็กซี่ EV7 ครับ"),
    (["สวัสดี", "หวัดดีครับ", "ทำอะไรได้บ้าง"],
     "สวัสดีครับ ถามเรื่องรถแท็กซี่ EV7 ได้เลย เช่น จำนวนรถในพื้นที่ ตำแหน่งรถ ระยะทาง ช่วงเวลาที่ผู้โดยสารเยอะ สถานีชาร์จ"),
]


# ── สร้าง + ตรวจกับข้อมูลจริง ───────────────────────────────────────────────
def fill_q(q, a):
    return q.format(p=ptypo(a["P"]) if "P" in a else "", d=typo(a["D"], 0.2) if "D" in a else "",
                    pl=plate_text(a["PL"]) if "PL" in a else "", day=a.get("DAYW", ""),
                    h=th_h(a["H"]) if "H" in a else "", h2=th_h(a["H2"]) if "H2" in a else "",
                    hr=a.get("HR", ""), spd=a.get("SPD", ""))


def call(sql):
    return "<tool_call>\n" + json.dumps({"name": "run_sql", "arguments": {"sql": sql}}, ensure_ascii=False) + "\n</tool_call>"


examples, dropped = [], 0
PER_Q = 12                                           # ตัวอย่างต่อสำนวน
for f in F:
    for qi, q in enumerate(f["qs"]):
        for _ in range(PER_Q):
            a = f["gen"]()
            if "PL" in a:
                a["K"] = pkey(a["PL"])
            if "H2" in a:
                a["H2m"] = a["H2"] - 1
            sql = f["sql"].format(**a)
            try:
                d = db.execute(sql).df()
                if d.empty or d.iloc[0].isna().all():
                    dropped += 1
                    continue
                ans = f["ans"](d, a)
            except Exception:
                dropped += 1
                continue
            uq = re.sub(r"\s+", " ", random.choice(PRE) + fill_q(q, a) + random.choice(POST)).strip()
            msgs = [{"role": "user", "content": uq},
                    {"role": "assistant", "content": call(re.sub(r"\s+", " ", sql))},
                    {"role": "tool", "content": _fmt(d)},
                    {"role": "assistant", "content": ans}]
            examples.append({"family": f["name"], "style": f"{f['name']}#{qi}", "messages": msgs})
for qi, (qs, ans) in enumerate(NO_SQL):
    for k, q in enumerate(qs):
        for _ in range(6):
            uq = re.sub(r"\s+", " ", random.choice(PRE) + q.format(pl=plate_text(random.choice(PLATES))) + random.choice(POST)).strip()
            examples.append({"family": f"nosql{qi}", "style": f"nosql{qi}#{k}",
                             "messages": [{"role": "user", "content": uq}, {"role": "assistant", "content": ans}]})

TRAIN_DF = pd.DataFrame(examples)
TRAIN_DF["q"] = TRAIN_DF.messages.map(lambda m: m[0]["content"])
TRAIN_DF = TRAIN_DF.drop_duplicates("q").reset_index(drop=True)

# แบ่ง test = 1 สำนวนต่อแบบคำถาม (ตัวสุดท้าย) → โมเดลไม่เคยเห็นสำนวนนี้ตอนเทรน
last_style = TRAIN_DF.groupby("family")["style"].agg(lambda s: sorted(s.unique())[-1])
TRAIN_DF["split"] = ["test" if s == last_style[f] else "train" for f, s in zip(TRAIN_DF.family, TRAIN_DF["style"])]

out_path = DRIVE / "taxi_sql_train.jsonl"
with open(out_path, "w", encoding="utf-8") as fh:
    for r in TRAIN_DF.itertuples():
        fh.write(json.dumps({"split": r.split, "family": r.family, "messages": r.messages}, ensure_ascii=False) + "\n")
print(f"ตัวอย่าง {len(TRAIN_DF):,} ข้อ ({len(F)} แบบที่ใช้ SQL + {len(NO_SQL)} แบบไม่ใช้ SQL) · ทิ้ง {dropped} ข้อที่ SQL ไม่ผ่าน")
print(TRAIN_DF.split.value_counts().to_dict(), "→", out_path)
for r in TRAIN_DF.sample(3, random_state=1).itertuples():
    print("\nQ:", r.messages[0]["content"])
    for m in r.messages[1:]:
        print(f"  [{m['role']}]", m["content"][:220].replace("\n", " "))

## 16 · เทรน Qwen (LoRA) → วัดผลก่อน/หลัง → ใช้ใน agent

- โมเดลตั้งต้น `Qwen2.5-7B-Instruct` (4-bit) · LoRA r=16 · 2 รอบ · สอนเฉพาะคำพูดของ assistant (SQL + คำตอบ)
- วัดผล: ให้โมเดลเขียน SQL ข้อสอบ 120 ข้อ → รันจริง → ถือว่าถูกเมื่อผลตรงเฉลยทุกตัว · แสดงก่อน/หลังเทรน + แบบคำถามที่ยังอ่อน
- เวลาเทรนบน T4 ~1–2 ชม. (ประมาณ — ดู log) · Kaggle ตัด session ที่ 12 ชม. · อย่าปิดแท็บระหว่างเทรน
- เสร็จแล้ว `agent()` ใช้โมเดลที่เทรนอัตโนมัติ (prompt สั้นกว่า → ตอบเร็วขึ้น) · adapter อยู่ที่ `/kaggle/working/lora_sql` → ดาวน์โหลดจาก Output

In [ ]:
# ── 16 · เทรน Qwen ด้วยชุดข้อมูลจาก cell 15 (LoRA) → วัดผล → เอาไปใช้แทนโมเดลใน agent ──
# ต้องรัน 0–7 · 13 · 15 ก่อน · GPU T4 x2 · เวลาเทรนขึ้นกับจำนวนตัวอย่าง (ดูตัวเลข ETA ตอนรัน)
#
# สอนอะไร: อ่านคำถามไทย → เขียน SQL ที่ถูก (ตามตารางของเรา) → อ่านผล → ตอบสั้น ๆ
# ทำไมใช้ LoRA: ไม่แตะน้ำหนักเดิม 7 พันล้านตัว · เทรนแค่ชั้นเล็กที่แปะเพิ่ม → พอ GPU ฟรี · ไฟล์ผลลัพธ์หลักสิบ MB
import gc, json, math, re, subprocess, sys, time
import pandas as pd
import torch

FT_BASE = "Qwen/Qwen2.5-7B-Instruct"     # โมเดลตั้งต้น · เร็วกว่า: "Qwen/Qwen2.5-3B-Instruct"
EPOCHS = 2
LR = 1e-4
LORA_R = 16
MAX_LEN = 1536                           # ตัดตัวอย่างที่ยาวเกิน (token) · ปกติ ~800
EVAL_N = 120                             # จำนวนข้อสอบ (test) ที่ใช้วัดผล · ทั้งหมด = None (นาน)
EVAL_BASE = True                         # วัดโมเดลก่อนเทรนด้วย (ไว้เทียบ) · รันซ้ำแล้วรู้ผลแล้ว ตั้ง False ประหยัด ~20 นาที
OUT_DIR = DRIVE / "lora_sql"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "bitsandbytes", "accelerate"], check=True)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel

# ── 0) เคลียร์ GPU (โมเดลจาก cell อื่น) ──────────────────────────────────
for v in ("amodel", "cmodel", "model", "base", "trainer", "ftmodel", "fbase"):   # รวมของรอบก่อนที่ค้าง
    if v in globals():
        del globals()[v]
gc.collect(); torch.cuda.empty_cache()

# ── 1) ข้อมูล ──────────────────────────────────────────────────────────
rows = [json.loads(l) for l in open(DRIVE / "taxi_sql_train.jsonl", encoding="utf-8")]
train_rows = [r for r in rows if r["split"] == "train"]
test_rows = [r for r in rows if r["split"] == "test"]
print(f"train {len(train_rows):,} · test {len(test_rows):,}")

ftok = AutoTokenizer.from_pretrained(FT_BASE)
ftok.pad_token = ftok.pad_token or ftok.eos_token


def render(msgs, gen=False):
    return ftok.apply_chat_template([{"role": "system", "content": SYSTEM_FT}] + msgs, tools=TOOLS_FT,
                                    tokenize=False, add_generation_prompt=gen)


def encode(msgs):
    """token ทั้งบทสนทนา · labels = -100 ทุกส่วนยกเว้นคำตอบของ assistant (สอนเฉพาะสิ่งที่โมเดลต้องพูด)"""
    ids = ftok(render(msgs), add_special_tokens=False)["input_ids"]
    labels = [-100] * len(ids)
    for j, m in enumerate(msgs):
        if m["role"] != "assistant":
            continue
        a = len(ftok(render(msgs[:j], gen=True), add_special_tokens=False)["input_ids"])
        b = len(ftok(render(msgs[:j + 1]), add_special_tokens=False)["input_ids"])
        labels[a:b] = ids[a:b]
    return {"input_ids": ids[:MAX_LEN], "labels": labels[:MAX_LEN]}


enc_train = [encode(r["messages"]) for r in train_rows]
lens = pd.Series([len(e["input_ids"]) for e in enc_train])
print(f"ความยาว token: เฉลี่ย {lens.mean():.0f} · สูงสุด {lens.max()} · ถูกตัด {int((lens >= MAX_LEN).sum())} ข้อ")


class DS(torch.utils.data.Dataset):
    def __init__(self, x): self.x = x
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i]


def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    pad = lambda s, v: s + [v] * (m - len(s))
    return {"input_ids": torch.tensor([pad(b["input_ids"], ftok.pad_token_id) for b in batch]),
            "labels": torch.tensor([pad(b["labels"], -100) for b in batch]),
            "attention_mask": torch.tensor([pad([1] * len(b["input_ids"]), 0) for b in batch])}


# ── 2) โหลดโมเดลตั้งต้น 4-bit + ติด LoRA ───────────────────────────────────
mem = {i: f"{int(torch.cuda.get_device_properties(i).total_memory / 2**30) - 2}GiB" for i in range(torch.cuda.device_count())}
fbase = AutoModelForCausalLM.from_pretrained(
    FT_BASE, device_map="auto", max_memory=mem, dtype=torch.float16,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16))
# ไม่ใช้ prepare_model_for_kbit_training — มันแปลง embedding/lm_head เป็น fp32 (+2 GB ต่อชั้น) จน T4 เต็ม
fbase.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
fbase.enable_input_require_grads()
fbase.config.use_cache = False
ftmodel = get_peft_model(fbase, LoraConfig(
    r=LORA_R, lora_alpha=LORA_R * 2, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
ftmodel.print_trainable_parameters()


# ── 3) วัดผล: ให้โมเดลเขียน SQL ข้อสอบ → รันจริง → เทียบผลกับเฉลย ───────────────
TOOL_RE_FT = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)


def first_turn(msgs_user):
    x = ftok(render(msgs_user, gen=True), return_tensors="pt").to(ftmodel.device)
    with torch.no_grad():
        out = ftmodel.generate(**x, max_new_tokens=300, do_sample=False, pad_token_id=ftok.pad_token_id, use_cache=True)
    return ftok.decode(out[0][x["input_ids"].shape[1]:], skip_special_tokens=True)


def judge(r, text):
    gold = r["messages"]
    m = TOOL_RE_FT.search(text)
    if len(gold) == 2:                                  # ข้อที่ไม่ควรเรียก SQL
        return m is None
    if not m:
        return False
    try:
        sql = json.loads(m.group(1))["arguments"]["sql"]
    except Exception:
        return False
    return tool_run_sql(sql).strip() == gold[2]["content"].strip()   # ผลตรงเฉลยทุกตัว


def evaluate(label):
    import random as _r
    _r.seed(0)
    sample = test_rows if EVAL_N is None else _r.sample(test_rows, min(EVAL_N, len(test_rows)))
    ok, t0, per = 0, time.time(), {}
    ftmodel.eval()
    for r in sample:
        good = judge(r, first_turn(r["messages"][:1]))
        ok += good
        per.setdefault(r["family"], []).append(good)
    acc = ok / len(sample)
    print(f"[{label}] ถูก {ok}/{len(sample)} = {acc:.0%} · {(time.time() - t0) / len(sample):.1f} วิ/ข้อ")
    return acc, pd.Series({k: sum(v) / len(v) for k, v in per.items()}).sort_values()


if EVAL_BASE:
    with ftmodel.disable_adapter():
        base_acc, base_per = evaluate("ก่อนเทรน")

# ── 4) เทรน ────────────────────────────────────────────────────────────
steps = math.ceil(len(enc_train) / 8) * EPOCHS
print(f"เริ่มเทรน {EPOCHS} รอบ · {steps} steps · ดู loss ลดลงเรื่อย ๆ = กำลังเรียน")
trainer = Trainer(
    model=ftmodel, train_dataset=DS(enc_train), data_collator=collate,
    args=TrainingArguments(output_dir="/kaggle/working/ft_ckpt", per_device_train_batch_size=1, gradient_accumulation_steps=8,
                           num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type="cosine", warmup_steps=max(1, int(0.05 * steps)),   # transformers 5 เลิกใช้ warmup_ratio
                           fp16=True, logging_steps=10, save_strategy="no", report_to=[], optim="paged_adamw_8bit",
                           gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
                           remove_unused_columns=False))
t0 = time.time()
res = trainer.train()
print(f"เทรนเสร็จ {(time.time() - t0) / 60:.0f} นาที · loss เฉลี่ย {res.metrics['train_loss']:.3f}")
ftmodel.save_pretrained(str(OUT_DIR)); ftok.save_pretrained(str(OUT_DIR))
print("บันทึก adapter →", OUT_DIR, "(ดาวน์โหลดจากแท็บ Output แล้วอัปเข้า dataset เพื่อใช้ครั้งหน้า)")

# ── 5) วัดผลหลังเทรน ─────────────────────────────────────────────────────
ft_acc, ft_per = evaluate("หลังเทรน")
if EVAL_BASE:
    cmp = pd.DataFrame({"ก่อน": base_per, "หลัง": ft_per}).fillna(0).sort_values("หลัง")
    print(f"\nก่อนเทรน {base_acc:.0%} → หลังเทรน {ft_acc:.0%}\nแบบคำถามที่ยังอ่อน (หลังเทรน):")
    print((cmp.head(8) * 100).round().astype(int).astype(str) + "%")

# ── 6) ใช้กับ agent (cell 13): สลับโมเดล + prompt สั้น ─────────────────────────
ftmodel.eval()
fbase.config.use_cache = True                       # ปิดไว้ตอนเทรน · เปิดคืนให้ตอบเร็ว
amodel, atok = ftmodel, ftok
SYSTEM_A, TOOLS_A = SYSTEM_FT, TOOLS_FT
agent_history.clear()
print('\nพร้อมใช้: agent("คำถาม", show_steps=True) ตอนนี้ใช้โมเดลที่เทรนแล้ว')
for q in ["ตอนนี้ภูเก็ตมีรถกี่คัน ว่างกี่คัน มีผู้โดยสารกี่คัน", "เขตไหนในกทมคนเรียกรถเยอะสุด", "ถนนสีลมมีรถกี่คัน"]:
    agent_history.clear()
    print("คุณ:", q); print("บอท:", agent(q, show_steps=True)); print()

### 16b · ใช้ adapter ที่เทรนไว้แล้ว (ครั้งถัดไป ไม่ต้องเทรนใหม่)

อัปโหลดโฟลเดอร์ `lora_sql` เข้า dataset → รัน 0–7 → 13 (`LOAD_AGENT_MODEL = False`) → 15 → cell นี้ → `agent(...)`

In [ ]:
# ── 16b · ใช้ adapter ที่เทรนไว้แล้ว (ไม่ต้องเทรนใหม่) ──
# ลำดับ: 0–7 → 13 (ตั้ง AGENT_LLM เป็นรุ่นเดียวกับ FT_BASE หรือข้ามการโหลดก็ได้) → 15 → cell นี้
# ต้องอัปโหลดโฟลเดอร์ lora_sql (จากแท็บ Output ของรอบที่เทรน) เข้า dataset ก่อน
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

FT_BASE = "Qwen/Qwen2.5-7B-Instruct"     # ต้องตรงกับตอนเทรน
_cfg = [p for p in INPUT.rglob("adapter_config.json") if "lora_sql" in str(p)] + list(DRIVE.glob("lora_sql/adapter_config.json"))
assert _cfg, "ไม่พบ lora_sql — อัปโหลดเข้า dataset แล้ว Add Input"
for v in ("amodel", "cmodel", "model", "base"):
    if v in globals():
        del globals()[v]
gc.collect(); torch.cuda.empty_cache()

atok = AutoTokenizer.from_pretrained(FT_BASE)
_b = AutoModelForCausalLM.from_pretrained(
    FT_BASE, device_map="auto", dtype=torch.float16,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16))
amodel = PeftModel.from_pretrained(_b, str(_cfg[0].parent)).eval()
SYSTEM_A, TOOLS_A = SYSTEM_FT, TOOLS_FT
agent_history.clear()
print("ใช้โมเดลที่เทรนแล้ว:", _cfg[0].parent)